# SeaFour Retrieval Engine Submission Notebook

This notebook is written as a reproducible engineering report for the retrieval competition. It keeps the original experimental intent of the submission notebook, but reorganizes the work into a clearer pipeline: data loading, preprocessing, index construction, retrieval, evaluation, and submission generation.

The notebook implements three first-stage retrieval methods:
- **TF-IDF** for a lightweight sparse lexical baseline
- **BM25+** for stronger lexical ranking with length normalization
- **Embedding retrieval** for dense semantic search with Sentence-Transformers

The default submission path still uses **embedding retrieval** plus the existing category classifier, because that is the strongest practical default for this dataset.

If a fresh environment is missing packages, install them before running the notebook:

```python
%pip install -q rank_bm25 sentence-transformers scikit-learn
```


## Retrieval Engine Overview

The end-to-end pipeline in this notebook is:

1. **Load competition files**: documents, train queries, test queries, training ground truth, and the sample submission template.
2. **Normalize text consistently**: merge the relevant text fields, standardize separators and whitespace, lowercase the text, and build a single `content` field for each document and query.
3. **Build retrieval artifacts**:
   - TF-IDF document-term matrix for lexical cosine search
   - BM25+ index over tokenized documents
   - Dense document embeddings for semantic retrieval
4. **Retrieve top-k candidates** for each query using the chosen first-stage retriever.
5. **Score the system offline** on the training queries using Recall@K, Precision@K, MRR@K, and category accuracy.
6. **Generate the final Kaggle submission** for the test queries.

This notebook uses direct first-stage retrieval only: retrieve top-k documents, evaluate on the training queries, and write the submission directly from the selected retriever.


## Why Embeddings Are the Primary Retrieval Method

Embedding retrieval is the best default method in this notebook because it captures **semantic similarity**, not only exact token overlap.

In practice, embeddings win here when:
- the query paraphrases the relevant document instead of reusing the same words
- users describe the same issue with different wording, abbreviations, or syntax
- the corpus is heterogeneous and contains multiple technical domains with overlapping vocabulary
- retrieval needs to stay robust when queries and documents use different wording for the same issue

Why embeddings are stronger than the lexical baselines in this notebook:
- **Better robustness to wording mismatch**: a query about a concept can still retrieve documents that do not share the same surface form.
- **Higher recall under semantic mismatch**: this matters a lot for support-style text, forum questions, and troubleshooting descriptions.
- **Better candidate generation**: dense retrieval is often the strongest first-stage retriever when wording mismatch is common.
- **More stable default behavior** when queries and relevant documents do not use the exact same vocabulary.

Honest limits of embeddings:
- they require more compute and more memory than simple lexical methods
- they can retrieve items that are semantically related but still not truly relevant
- they are not always best for **exact-match retrieval**, especially for rare identifiers, product codes, version strings, names, or error messages

In those exact-match cases, TF-IDF or BM25+ can still be valuable because lexical overlap is the signal you care about.


## Choosing `top_k`

`top_k` is the number of documents returned per query by the first-stage retriever.

Why `top_k` matters:
- **Too small**: the retriever may miss relevant documents, which hurts recall.
- **Too large**: recall usually improves, but precision drops, latency grows, and memory use increases.

Practical tradeoffs:

| Setting | Usually helps | Usually hurts |
|---|---|---|
| Small `top_k` | Precision, latency, memory footprint | Recall |
| Large `top_k` | Recall | Precision, latency, memory footprint |

Metric impact:
- **Recall@K** usually increases as `top_k` grows because more relevant documents can appear in the candidate list.
- **Precision@K** often decreases because the tail of the ranking includes more non-relevant documents.
- **MRR@K** changes less once the first relevant document is already near the top.
- **Latency** increases because more scores must be computed or sorted.
- **Memory** increases when storing larger score blocks and larger submission payloads.

In this notebook, `top_k` is also the final number of retrieved documents used for evaluation or submission.


## When to Use Baseline / Lexical / Embedding Retrieval

Use the method that matches the failure mode you expect most often.

| Method | Prefer it when | Less suitable when |
|---|---|---|
| **TF-IDF** | You want a simple, fast baseline; documents are short; overlap in rare terms is informative | Queries and documents use different phrasing or synonyms |
| **BM25+** | Exact words matter, document lengths vary, and you want a stronger lexical baseline than TF-IDF | Semantic mismatch is common |
| **Embeddings** | Queries paraphrase documents, recall matters, and the corpus is heterogeneous | You are extremely latency-constrained or exact IDs / codes dominate relevance |

Practical selection guidance:
- **Small datasets**: TF-IDF or BM25+ may be good enough and are easier to debug.
- **Large datasets**: embeddings often pay off because semantic mismatch becomes more common and lexical recall becomes brittle.
- **High semantic mismatch**: prefer embeddings.
- **Strict latency budget**: prefer lexical methods unless dense retrieval is well optimized or precomputed.
- **Recall more important than precision**: prefer embeddings and a larger candidate pool.
- **Precision at very small K matters more**: BM25+ can be competitive when exact wording is highly reliable.


In [ ]:
# Imports, paths, and experiment configuration.
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, Sequence, TypedDict
import csv
import hashlib
import json
import os
import pickle
import re
import time

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 5)

ModelName = Literal["tfidf", "bm25", "embedding"]


class RetrievalResult(TypedDict):
    """One ranked retrieval result for a single query."""

    query_id: str
    relevant_docs: list[str]


class GroundTruthEntry(TypedDict):
    """Ground-truth annotations for a single training query."""

    relevant_doc_ids: set[str]
    total_relevant_docs: int
    category: str | None


@dataclass(frozen=True)
class RuntimePaths:
    """Resolved filesystem locations for the active notebook runtime."""

    runtime_env: Literal["colab", "kaggle", "local"]
    project_dir: Path
    work_dir: Path
    data_dir: Path
    cache_dir: Path
    output_path: Path


def detect_runtime_environment() -> Literal["colab", "kaggle", "local"]:
    """Detect whether the notebook runs in Colab, Kaggle, or a local environment."""
    try:
        import google.colab  # type: ignore  # noqa: F401

        return "colab"
    except Exception:
        if Path("/kaggle/input").exists():
            return "kaggle"
        return "local"


def find_kaggle_data_dir() -> Path | None:
    """Search `/kaggle/input` for the folder that contains the competition JSON files."""
    for dirname, _, filenames in os.walk("/kaggle/input"):
        if "docs.json" in filenames:
            return Path(dirname)
    return None


def find_colab_project_dir(project_name: str = "retrieval_project") -> Path | None:
    """Locate the project folder on Google Drive when running in Colab."""
    drive_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    for drive_root in drive_candidates:
        if not drive_root.exists():
            continue

        direct_candidate = drive_root / project_name
        if (direct_candidate / "data" / "docs.json").exists():
            return direct_candidate

        for candidate in drive_root.rglob(project_name):
            if candidate.is_dir() and (candidate / "data" / "docs.json").exists():
                return candidate

    return None


def find_local_project_data_dir() -> Path | None:
    """Find the `data/` directory near the current working directory."""
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        data_dir = root / "data"
        if (data_dir / "docs.json").exists():
            return data_dir
    return None


def resolve_runtime_paths(output_filename: str = "solutions_SeaFour.csv") -> RuntimePaths:
    """Resolve project, data, cache, and output paths for the active runtime."""
    runtime_env = detect_runtime_environment()
    project_dir: Path | None = None
    work_dir = Path.cwd()

    if runtime_env == "colab":
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        project_dir = find_colab_project_dir("retrieval_project")
        if project_dir is None:
            raise FileNotFoundError(
                "Google Drive is mounted, but `retrieval_project/data/docs.json` was not found."
            )
        os.chdir(project_dir)
        work_dir = project_dir
        data_dir = project_dir / "data"
    elif runtime_env == "kaggle":
        data_dir = find_kaggle_data_dir()
        if data_dir is None:
            raise FileNotFoundError(
                "Kaggle environment detected, but `docs.json` was not found under /kaggle/input."
            )
        project_dir = Path.cwd()
    else:
        data_dir = find_local_project_data_dir()
        if data_dir is None:
            raise FileNotFoundError(
                "Could not find `data/docs.json` near the current working directory."
            )
        project_dir = data_dir.parent
        work_dir = project_dir
        os.chdir(work_dir)

    cache_dir = work_dir / "cache"
    output_path = work_dir / output_filename
    return RuntimePaths(
        runtime_env=runtime_env,
        project_dir=project_dir or work_dir,
        work_dir=work_dir,
        data_dir=data_dir,
        cache_dir=cache_dir,
        output_path=output_path,
    )


# High-level experiment settings.
FINAL_MODEL: ModelName = "embedding"
EVALUATION_MODELS: tuple[ModelName, ...] = ("embedding",)
EVALUATION_TOP_KS = [ 7500 ]#tuple[int, ...] = (7_500)#,12_500, 75_000)
SUBMIT_TOP_K = 7_500
ENABLE_CATEGORY_FILTER = True
DOMINANT_CATEGORY_TOP_N = 20
ENABLE_CROSS_ENCODER_RERANK = True
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"
CROSS_ENCODER_EPOCHS = 5
CROSS_ENCODER_BATCH_SIZE = 32
CROSS_ENCODER_MAX_LENGTH = 256
CROSS_ENCODER_MAX_POSITIVES_PER_QUERY = 4
CROSS_ENCODER_NEGATIVES_PER_POSITIVE = 4
CROSS_ENCODER_TRAIN_QUERY_LIMIT = 327
CROSS_ENCODER_HARD_NEGATIVE_TOP_K = 200
CROSS_ENCODER_RANDOM_SEED = 42
CROSS_ENCODER_RERANK_TOP_M = 150#20
CROSS_ENCODER_ZSCORE_WEIGHT = 1#0.75
CROSS_ENCODER_INFER_BATCH_SIZE = 64
CROSS_ENCODER_FP16 = True  # Use FP16 on GPU for faster inference
CROSS_ENCODER_CATEGORY_BONUS = 2.0  # Added to category score when query and doc categories match
ENABLE_CROSS_ENCODER_CACHE = True

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 256
EMBEDDING_QUERY_CHUNK_SIZE = 32

DOCUMENT_TEXT_COLUMNS: tuple[str, ...] = ("title", "text", "tags")
RETRIEVAL_QUERY_COLUMNS: tuple[str, ...] = ("title", "text")
CLASSIFIER_QUERY_COLUMNS: tuple[str, ...] = ("title", "text", "tags")
CLASSIFIER_USE_QUERY_TAGS = True

NORMALIZATION_CONFIG = {
    "lowercase": True,
    "replace_separators": True,
    "separator_chars": "-_/",
    "collapse_whitespace": True,
    "strip": True,
}
TOKEN_PATTERN = r"[a-z0-9]+"

TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
}
CLASSIFIER_TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
}
BM25_CONFIG = {
    "k1": 1.5,
    "b": 0.75,
    "delta": 1.0,
}

ENABLE_EMBEDDING_CACHE = True
ENABLE_CLASSIC_CACHE = True

PATHS = resolve_runtime_paths()
MODEL_CACHE_DIR = PATHS.cache_dir / "sentence_transformers"
EMBEDDING_CACHE_DIR = PATHS.cache_dir / "embeddings"
TFIDF_CACHE_DIR = PATHS.cache_dir / "tfidf"
BM25_CACHE_DIR = PATHS.cache_dir / "bm25"
CLASSIFIER_CACHE_DIR = PATHS.cache_dir / "classifier"
CROSS_ENCODER_CACHE_DIR = PATHS.cache_dir / "cross_encoder"

for cache_path in [
    PATHS.cache_dir,
    MODEL_CACHE_DIR,
    EMBEDDING_CACHE_DIR,
    TFIDF_CACHE_DIR,
    BM25_CACHE_DIR,
    CLASSIFIER_CACHE_DIR,
    CROSS_ENCODER_CACHE_DIR,
]:
    cache_path.mkdir(parents=True, exist_ok=True)

print(f"Runtime environment: {PATHS.runtime_env}")
print(f"Project directory  : {PATHS.project_dir}")
print(f"Working directory  : {PATHS.work_dir}")
print(f"Data directory     : {PATHS.data_dir}")
print(f"Output path        : {PATHS.output_path}")


## Data Loading

This section loads the raw competition files and validates the expected schema early. The checks are intentionally strict so notebook failures happen close to the source of the problem instead of later in the retrieval pipeline.


In [ ]:
def require_columns(frame: pd.DataFrame, required_columns: Sequence[str], frame_name: str) -> None:
    """Raise a clear error if a dataframe is missing required columns."""
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{frame_name} is missing required columns: {missing_columns}")


def ensure_unique_ids(frame: pd.DataFrame, frame_name: str) -> None:
    """Ensure that the `id` column exists and does not contain duplicates."""
    require_columns(frame, ["id"], frame_name)
    if not frame["id"].astype(str).is_unique:
        raise ValueError(f"{frame_name} contains duplicate ids, which would break retrieval output mapping.")


def load_json_frame(path: Path, frame_name: str) -> pd.DataFrame:
    """Load one JSON competition file into a dataframe."""
    if not path.exists():
        raise FileNotFoundError(f"{frame_name} file not found: {path}")
    return pd.read_json(path)


docs_raw_df = load_json_frame(PATHS.data_dir / "docs.json", "Documents")
train_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_train.json", "Train queries")
test_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_test.json", "Test queries")
sample_submission_path = PATHS.data_dir / "submission.csv"
ground_truth_path = PATHS.data_dir / "qgts_train.json"

sample_submission_df = pd.read_csv(sample_submission_path)

require_columns(docs_raw_df, ["id", "title", "text", "tags", "category"], "Documents")
require_columns(train_queries_raw_df, ["id", "title", "text", "tags", "category"], "Train queries")
require_columns(test_queries_raw_df, ["id", "title", "text", "tags"], "Test queries")
require_columns(sample_submission_df, ["query_id", "relevant_doc_ids", "category"], "Sample submission")

ensure_unique_ids(docs_raw_df, "Documents")
ensure_unique_ids(train_queries_raw_df, "Train queries")
ensure_unique_ids(test_queries_raw_df, "Test queries")

print(f"Documents      : {len(docs_raw_df):,}")
print(f"Train queries  : {len(train_queries_raw_df):,}")
print(f"Test queries   : {len(test_queries_raw_df):,}")
print(f"Sample rows    : {len(sample_submission_df):,}")


## Preprocessing

Retrieval quality depends heavily on text normalization. The goal here is not aggressive linguistic processing; it is simple, consistent engineering hygiene. We create a single `content` field per row, keep the logic shared across methods, and preserve the original design choice that retrieval uses only query title and text, while the classifier may also use query tags.


In [ ]:
_TOKEN_RE = re.compile(TOKEN_PATTERN)
_WHITESPACE_RE = re.compile(r"\s+")
_SEPARATOR_RE = re.compile(f"[{re.escape(NORMALIZATION_CONFIG['separator_chars'])}]")


def value_to_text(value: Any) -> str:
    """Convert raw dataframe values into plain text.

    Args:
        value: A scalar, list-like, or missing value from a dataframe cell.

    Returns:
        A string representation suitable for text normalization.
    """
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(text: Any) -> str:
    """Apply the shared normalization policy used across retrievers.

    Args:
        text: Raw text or text-like content.

    Returns:
        Normalized text with separators replaced, whitespace collapsed, and casing standardized.
    """
    if text is None:
        cleaned_text = ""
    elif not isinstance(text, str) and pd.isna(text):
        cleaned_text = ""
    else:
        cleaned_text = str(text)

    if NORMALIZATION_CONFIG["replace_separators"]:
        cleaned_text = _SEPARATOR_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["lowercase"]:
        cleaned_text = cleaned_text.lower()
    if NORMALIZATION_CONFIG["collapse_whitespace"]:
        cleaned_text = _WHITESPACE_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["strip"]:
        cleaned_text = cleaned_text.strip()
    return cleaned_text


def build_content_frame(frame: pd.DataFrame, text_columns: Sequence[str]) -> pd.DataFrame:
    """Build a dataframe with a normalized `content` column.

    Args:
        frame: Input dataframe that must contain `id` and the requested text columns.
        text_columns: Columns to concatenate into the normalized content field.

    Returns:
        A copy of the input dataframe with string ids and a new `content` column.
    """
    require_columns(frame, ["id"], "Input frame")
    output_frame = frame.copy()
    text_parts: list[list[str]] = []

    for column in text_columns:
        if column in output_frame.columns:
            text_parts.append(output_frame[column].map(value_to_text).tolist())
        else:
            text_parts.append([""] * len(output_frame))

    merged_text = [" ".join(parts) for parts in zip(*text_parts)]
    output_frame["content"] = [normalize_text(text) for text in merged_text]
    output_frame["id"] = output_frame["id"].astype(str)
    return output_frame


def tokenize(text: str) -> list[str]:
    """Tokenize normalized text for lexical retrieval."""
    return _TOKEN_RE.findall(normalize_text(text))


def build_query_classifier_frame(query_frame: pd.DataFrame, include_tags: bool = CLASSIFIER_USE_QUERY_TAGS) -> pd.DataFrame:
    """Build the text view used by the category classifier.

    Args:
        query_frame: Raw query dataframe.
        include_tags: Whether to include the `tags` column when available.

    Returns:
        A dataframe with a classifier-oriented `content` column.
    """
    columns = ["title", "text"]
    if include_tags and "tags" in query_frame.columns:
        columns.append("tags")
    return build_content_frame(query_frame, columns)


docs_df = build_content_frame(docs_raw_df, DOCUMENT_TEXT_COLUMNS)
train_queries_df = build_content_frame(train_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)
test_queries_df = build_content_frame(test_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)

docs_classifier_df = docs_df[["id", "content", "category"]].copy()
train_queries_classifier_df = build_query_classifier_frame(train_queries_raw_df)
test_queries_classifier_df = build_query_classifier_frame(test_queries_raw_df)

if docs_df["content"].eq("").all():
    raise ValueError("All document content is empty after preprocessing. Check the source columns or normalization.")

print(f"Average document length (chars): {docs_df['content'].str.len().mean():.1f}")
print(f"Average train query length      : {train_queries_df['content'].str.len().mean():.1f}")
print(f"Average test query length       : {test_queries_df['content'].str.len().mean():.1f}")


## Dataset & Category Distribution Analysis

Visualises the document and query counts per category, highlights imbalance between them, and provides a coverage overview showing avg relevant docs per query from the ground truth.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from pathlib import Path

# ── helpers ──────────────────────────────────────────────────────────────────
def _bar_labels(ax, counts, total, fontsize=8):
    """Annotate horizontal bars with count + percentage."""
    for bar, cnt in zip(ax.patches, counts):
        pct = cnt / total * 100
        ax.text(
            bar.get_width() + total * 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{cnt:,}  ({pct:.1f}%)",
            va="center",
            ha="left",
            fontsize=fontsize,
        )

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle("Dataset & Category Distribution Analysis", fontsize=16, fontweight="bold", y=1.01)

# ── subplot 1: document distribution ─────────────────────────────────────────
ax1 = axes[0, 0]
doc_counts = docs_df["category"].value_counts().sort_values(ascending=True)
n_cats = len(doc_counts)
colors1 = plt.colormaps["viridis"](np.linspace(0.2, 0.9, n_cats))
doc_counts.plot(kind="barh", ax=ax1, color=colors1)
_bar_labels(ax1, doc_counts.values, doc_counts.sum())
ax1.set_title("Document distribution by category", fontweight="bold")
ax1.set_xlabel("Document count")
ax1.set_xlim(right=doc_counts.max() * 1.25)

# ── subplot 2: train query distribution ──────────────────────────────────────
ax2 = axes[0, 1]
qry_counts = train_queries_df["category"].value_counts().sort_values(ascending=True)
n_cats2 = len(qry_counts)
colors2 = plt.colormaps["coolwarm"](np.linspace(0.1, 0.9, n_cats2))
qry_counts.plot(kind="barh", ax=ax2, color=colors2)
_bar_labels(ax2, qry_counts.values, qry_counts.sum())
ax2.set_title("Train query distribution by category", fontweight="bold")
ax2.set_xlabel("Query count")
ax2.set_xlim(right=qry_counts.max() * 1.25)

# ── subplot 3: docs-per-query imbalance ratio ─────────────────────────────────
ax3 = axes[1, 0]
import pandas as pd
all_cats = sorted(set(doc_counts.index) | set(qry_counts.index))
ratio_data = {
    cat: doc_counts.get(cat, 0) / max(qry_counts.get(cat, 0), 1)
    for cat in all_cats
}
ratio_series = pd.Series(ratio_data).sort_values(ascending=True)
mean_ratio = ratio_series.mean()
ratio_series.plot(kind="barh", ax=ax3, color="steelblue")
ax3.axvline(mean_ratio, color="red", linestyle="--", linewidth=1.5, label=f"Mean = {mean_ratio:.1f}")
for bar, val in zip(ax3.patches, ratio_series.values):
    ax3.text(
        bar.get_width() + ratio_series.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center",
        ha="left",
        fontsize=8,
    )
ax3.legend(fontsize=9)
ax3.set_title("Docs per query ratio by category", fontweight="bold")
ax3.set_xlabel("Docs / query")
ax3.set_xlim(right=ratio_series.max() * 1.18)

# ── subplot 4: category coverage heatmap ─────────────────────────────────────
ax4 = axes[1, 1]

# ground_truth is dict[str, GroundTruthEntry] where GroundTruthEntry is a TypedDict
test_cat_counts = (
    test_queries_df["category"].value_counts()
    if "category" in test_queries_df.columns
    else pd.Series(dtype=int)
)

gt_by_cat: dict = {}
for entry in ground_truth.values():
    cat = entry["category"]
    gt_by_cat.setdefault(cat, []).append(len(entry["relevant_doc_ids"]))

heatmap_cats = sorted(all_cats)
col_labels = ["Total docs", "Train queries", "Avg rel docs/query"]
raw_matrix = np.zeros((len(heatmap_cats), len(col_labels)))
for r, cat in enumerate(heatmap_cats):
    raw_matrix[r, 0] = doc_counts.get(cat, 0)
    raw_matrix[r, 1] = qry_counts.get(cat, 0)
    rel_list = gt_by_cat.get(cat, [])
    raw_matrix[r, 2] = float(np.mean(rel_list)) if rel_list else 0.0

# normalise each column to [0, 1] for colour only
norm_matrix = np.zeros_like(raw_matrix, dtype=float)
for c in range(raw_matrix.shape[1]):
    col = raw_matrix[:, c]
    col_min, col_max = col.min(), col.max()
    if col_max > col_min:
        norm_matrix[:, c] = (col - col_min) / (col_max - col_min)
    else:
        norm_matrix[:, c] = 0.5

im = ax4.imshow(norm_matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax4.set_xticks(range(len(col_labels)))
ax4.set_xticklabels(col_labels, fontsize=9, fontweight="bold")
ax4.set_yticks(range(len(heatmap_cats)))
ax4.set_yticklabels(heatmap_cats, fontsize=8)

fmt_fns = [lambda v: f"{int(v):,}", lambda v: f"{int(v):,}", lambda v: f"{v:.1f}"]
for r in range(len(heatmap_cats)):
    for c in range(len(col_labels)):
        text_color = "white" if norm_matrix[r, c] > 0.6 else "black"
        ax4.text(
            c, r,
            fmt_fns[c](raw_matrix[r, c]),
            ha="center", va="center",
            fontsize=7.5, color=text_color,
        )
ax4.set_title("Category coverage overview", fontweight="bold")
fig.colorbar(im, ax=ax4, fraction=0.046, pad=0.04, label="Normalised value")

plt.tight_layout()

# save figure
report_dir = PATHS.work_dir / "report"
report_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(report_dir / "category_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# ── text summary ──────────────────────────────────────────────────────────────
total_docs = len(docs_df)
total_cats = len(all_cats)
top_doc_cat = doc_counts.index[-1]
top_doc_cnt = int(doc_counts.iloc[-1])
bot_doc_cat = doc_counts.index[0]
bot_doc_cnt = int(doc_counts.iloc[0])
imbalance = top_doc_cnt / max(bot_doc_cnt, 1)

print("=" * 65)
print(f"Total documents : {total_docs:,}")
print(f"Total categories: {total_cats}")
print()
print(f"Most  represented: {top_doc_cat!r:30s} {top_doc_cnt:>8,}  ({top_doc_cnt/total_docs*100:.1f}%)")
print(f"Least represented: {bot_doc_cat!r:30s} {bot_doc_cnt:>8,}  ({bot_doc_cnt/total_docs*100:.1f}%)")
print(f"Imbalance ratio (max/min): {imbalance:.1f}x")
print()
print(f"{'Category':<30} {'Train Q':>8} {'Test Q':>8} {'Avg rel docs/Q':>15}")
print("-" * 65)
for cat in sorted(all_cats):
    tq = int(qry_counts.get(cat, 0))
    tst = int(test_cat_counts.get(cat, 0)) if len(test_cat_counts) else 0
    rel_list = gt_by_cat.get(cat, [])
    avg_rel = float(np.mean(rel_list)) if rel_list else 0.0
    print(f"{cat:<30} {tq:>8,} {tst:>8,} {avg_rel:>15.1f}")

print()
sparse_cats = [c for c in all_cats if doc_counts.get(c, 0) < 1000]
dense_cats  = [c for c in all_cats if doc_counts.get(c, 0) > 50_000]
if sparse_cats:
    print(f"Categories with <1,000 docs  (potential hard cases) : {sparse_cats}")
else:
    print("No categories with <1,000 docs.")
if dense_cats:
    print(f"Categories with >50,000 docs (potential noise sources): {dense_cats}")
else:
    print("No categories with >50,000 docs.")
print("=" * 65)


In [ ]:
# Diagnostic: do any queries have relevant docs spanning multiple categories?
_id_to_cat = docs_df.set_index("id")["category"]
_multi_cat_queries = {}
for qid, entry in ground_truth.items():
    relevant_ids = entry["relevant_doc_ids"]
    cats = set(_id_to_cat.loc[_id_to_cat.index.isin(relevant_ids)].values)
    if len(cats) > 1:
        _multi_cat_queries[qid] = cats

if _multi_cat_queries:
    print(f"{len(_multi_cat_queries):,} / {len(ground_truth):,} queries have relevant docs in multiple categories:")
    for qid, cats in list(_multi_cat_queries.items())[:20]:
        print(f"  {qid}: {cats}")
    if len(_multi_cat_queries) > 20:
        print(f"  ... and {len(_multi_cat_queries) - 20} more")
else:
    print("No queries span multiple categories — hard category filtering is safe.")


## Retrieval Index Construction

The main engineering goal in this section is to make expensive work reusable. We cache document embeddings, TF-IDF artifacts, BM25 indexes, and the category classifier so repeated notebook runs do not recompute the same objects unnecessarily.

The dense retriever is the most expensive component, so caching and chunked scoring matter most there.


In [ ]:
_MODEL_MEMORY_CACHE: dict[str, Any] = {}
_ARRAY_MEMORY_CACHE: dict[str, np.ndarray] = {}
_OBJECT_MEMORY_CACHE: dict[str, Any] = {}


def _safe_component(value: Any) -> str:
    """Make a string safe for use in cache file names."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))


def _hash_payload(payload: dict[str, Any]) -> str:
    """Create a short deterministic hash for cache keys."""
    raw_payload = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha1(raw_payload.encode("utf-8")).hexdigest()[:16]


def _normalization_signature() -> str:
    """Fingerprint the preprocessing configuration used by the retrievers."""
    payload = {
        "normalization": NORMALIZATION_CONFIG,
        "token_pattern": TOKEN_PATTERN,
    }
    return _hash_payload(payload)


def _dataframe_fingerprint(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    """Fingerprint selected dataframe columns for cache invalidation."""
    hasher = hashlib.sha1()
    hasher.update(str(len(frame)).encode("utf-8"))
    for column in columns:
        hasher.update(column.encode("utf-8"))
        column_hash = pd.util.hash_pandas_object(frame[column].astype(str), index=False).values
        hasher.update(column_hash.tobytes())
    return hasher.hexdigest()[:16]


def _load_pickle(path: Path) -> Any:
    """Load a pickled artifact from disk."""
    with open(path, "rb") as handle:
        return pickle.load(handle)


def _save_pickle(path: Path, artifact: Any) -> None:
    """Persist an artifact to disk with the highest pickle protocol."""
    with open(path, "wb") as handle:
        pickle.dump(artifact, handle, protocol=pickle.HIGHEST_PROTOCOL)


def _load_sentence_model(model_name: str) -> Any:
    """Load a Sentence-Transformer model, preferring the local cache when available."""
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    if model_name in _MODEL_MEMORY_CACHE:
        return _MODEL_MEMORY_CACHE[model_name]

    safe_model_name = _safe_component(model_name)
    local_model_dir = MODEL_CACHE_DIR / safe_model_name
    if local_model_dir.exists():
        print(f"Loading model weights from cache: {local_model_dir}")
        model = SentenceTransformer(str(local_model_dir))
    else:
        print(f"Downloading model weights: {model_name}")
        model = SentenceTransformer(model_name)
        local_model_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(local_model_dir))
        print(f"Saved model weights to cache: {local_model_dir}")

    _MODEL_MEMORY_CACHE[model_name] = model
    return model


def _load_or_encode_embeddings(
    frame: pd.DataFrame,
    kind: str,
    model: Any,
    model_name: str,
    batch_size: int,
) -> np.ndarray:
    """Load cached embeddings or encode them once and persist the result.

    Args:
        frame: Dataframe containing `id` and normalized `content`.
        kind: Human-readable cache prefix such as `docs` or `queries_train`.
        model: Loaded Sentence-Transformer model.
        model_name: Model identifier used in cache keys.
        batch_size: Sentence-Transformer encoding batch size.

    Returns:
        A float32 matrix of L2-normalized embeddings.
    """
    signature = _dataframe_fingerprint(frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_name = f"{kind}_{_safe_component(model_name)}_{normalization_signature}_{signature}.npy"
    cache_path = EMBEDDING_CACHE_DIR / cache_name
    memory_key = str(cache_path.resolve())

    if memory_key in _ARRAY_MEMORY_CACHE:
        return _ARRAY_MEMORY_CACHE[memory_key]

    if ENABLE_EMBEDDING_CACHE and cache_path.exists():
        print(f"Loading {kind} embeddings from cache: {cache_path.name}")
        embeddings = np.load(cache_path)
    else:
        print(f"Encoding {len(frame):,} {kind} rows...")
        embeddings = model.encode(
            frame["content"].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if ENABLE_EMBEDDING_CACHE:
            np.save(cache_path, embeddings)
            print(f"Saved {kind} embeddings to cache: {cache_path.name}")

    _ARRAY_MEMORY_CACHE[memory_key] = embeddings
    return embeddings


def _tfidf_param_candidates() -> list[dict[str, Any]]:
    """Return TF-IDF parameter settings including a safe fallback for very small corpora."""
    candidates = [dict(TFIDF_CONFIG)]
    min_df = TFIDF_CONFIG.get("min_df", 1)
    if isinstance(min_df, int) and min_df > 1:
        fallback = dict(TFIDF_CONFIG)
        fallback["min_df"] = 1
        candidates.append(fallback)
    return candidates


def build_or_load_tfidf_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF vectorizer and document matrix."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    doc_ids = docs_frame["id"].to_numpy()

    for params in _tfidf_param_candidates():
        cache_key = _hash_payload(
            {
                "docs_signature": docs_signature,
                "normalization_signature": normalization_signature,
                "tfidf_params": params,
            }
        )
        cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
        memory_key = str(cache_path.resolve())

        if memory_key in _OBJECT_MEMORY_CACHE:
            return _OBJECT_MEMORY_CACHE[memory_key]
        if ENABLE_CLASSIC_CACHE and cache_path.exists():
            print(f"Loading TF-IDF artifacts from cache: {cache_path.name}")
            artifacts = _load_pickle(cache_path)
            _OBJECT_MEMORY_CACHE[memory_key] = artifacts
            return artifacts

    params = dict(TFIDF_CONFIG)
    vectorizer = TfidfVectorizer(**params)
    try:
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])
    except ValueError as err:
        if "After pruning, no terms remain" not in str(err) or params.get("min_df", 1) == 1:
            raise
        params["min_df"] = 1
        vectorizer = TfidfVectorizer(**params)
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])

    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "tfidf_params": params,
        }
    )
    cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    artifacts = {
        "vectorizer": vectorizer,
        "doc_vectors": doc_vectors,
        "doc_ids": doc_ids,
        "params": params,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved TF-IDF artifacts to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_bm25_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached BM25+ index."""
    try:
        from rank_bm25 import BM25Plus
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `rank_bm25`. Install it with `%pip install rank_bm25`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "bm25_params": BM25_CONFIG,
        }
    )
    cache_path = BM25_CACHE_DIR / f"bm25_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading BM25 index from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    tokenized_corpus = [tokenize(text) for text in docs_frame["content"]]
    bm25 = BM25Plus(tokenized_corpus, **BM25_CONFIG)
    artifacts = {
        "bm25": bm25,
        "doc_ids": docs_frame["id"].to_numpy(),
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved BM25 index to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_category_classifier(train_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF + LinearSVC category classifier."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    require_columns(train_frame, ["id", "content", "category"], "Classifier training data")
    train_signature = _dataframe_fingerprint(train_frame, ["id", "content", "category"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "train_signature": train_signature,
            "normalization_signature": normalization_signature,
            "classifier_tfidf_params": CLASSIFIER_TFIDF_CONFIG,
        }
    )
    cache_path = CLASSIFIER_CACHE_DIR / f"category_classifier_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading category classifier from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    vectorizer = TfidfVectorizer(**CLASSIFIER_TFIDF_CONFIG)
    train_vectors = vectorizer.fit_transform(train_frame["content"])
    classifier = LinearSVC()
    classifier.fit(train_vectors, train_frame["category"].astype(str))

    artifacts = {
        "vectorizer": vectorizer,
        "classifier": classifier,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved category classifier to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


## Retrieval Functions

The retrieval code is intentionally separated from index construction. That keeps the notebook easier to reason about and makes it straightforward to compare methods fairly.

The main performance changes in this refactor are:
- use **partial top-k selection** with `np.argpartition` instead of sorting every score vector fully
- score embedding queries in **chunks** to avoid creating unnecessarily large dense matrices
- reuse the **same max-k ranking** during offline sweeps and truncate it for smaller K values


In [ ]:
def validate_pipeline_settings(document_count: int) -> None:
    """Validate global retrieval and submission settings."""
    if FINAL_MODEL not in {"tfidf", "bm25", "embedding"}:
        raise ValueError(f"Unknown FINAL_MODEL: {FINAL_MODEL}")
    if any(model_name not in {"tfidf", "bm25", "embedding"} for model_name in EVALUATION_MODELS):
        raise ValueError(f"Unknown model in EVALUATION_MODELS: {EVALUATION_MODELS}")
    if document_count <= 0:
        raise ValueError("The document collection is empty.")
    if SUBMIT_TOP_K <= 0:
        raise ValueError("SUBMIT_TOP_K must be positive.")


def top_k_indices(score_vector: np.ndarray, top_k: int) -> np.ndarray:
    """Return indices of the top-k scores in descending order.

    This uses `np.argpartition` to avoid a full sort when only the largest values are needed.
    """
    if top_k <= 0:
        raise ValueError("top_k must be positive.")

    capped_top_k = min(top_k, score_vector.shape[0])
    if capped_top_k == score_vector.shape[0]:
        return np.argsort(score_vector)[::-1]

    candidate_indices = np.argpartition(score_vector, -capped_top_k)[-capped_top_k:]
    sorted_candidates = candidate_indices[np.argsort(score_vector[candidate_indices])[::-1]]
    return sorted_candidates


def truncate_results(results: list[RetrievalResult], top_k: int) -> list[RetrievalResult]:
    """Truncate a ranked result list to a smaller K without recomputing scores."""
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    return [
        {
            "query_id": result["query_id"],
            "relevant_docs": result["relevant_docs"][:top_k],
        }
        for result in results
    ]


def progress_interval(total_items: int, target_updates: int = 5) -> int:
    """Choose a lightweight logging interval for progress messages."""
    return max(1, total_items // max(1, target_updates))


def prepare_retriever(model_name: ModelName, docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Prepare and cache the artifacts required by one retriever."""
    if model_name == "tfidf":
        return build_or_load_tfidf_index(docs_frame)
    if model_name == "bm25":
        return build_or_load_bm25_index(docs_frame)
    if model_name == "embedding":
        model = _load_sentence_model(EMBEDDING_MODEL_NAME)
        doc_embeddings = _load_or_encode_embeddings(
            docs_frame,
            kind="docs",
            model=model,
            model_name=EMBEDDING_MODEL_NAME,
            batch_size=EMBEDDING_BATCH_SIZE,
        )
        return {
            "model": model,
            "doc_embeddings": doc_embeddings,
            "doc_ids": docs_frame["id"].to_numpy(),
        }
    raise ValueError(f"Unknown model: {model_name}")


def run_tfidf_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run sparse lexical retrieval with TF-IDF cosine similarity."""
    artifacts = prepared_artifacts or build_or_load_tfidf_index(docs_frame)
    vectorizer = artifacts["vectorizer"]
    doc_vectors = artifacts["doc_vectors"]
    doc_ids = artifacts["doc_ids"]
    query_vectors = vectorizer.transform(queries_frame["content"])
    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    log_every = progress_interval(len(query_ids))

    print(
        f"  [TF-IDF] vectorized {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, query_id in enumerate(query_ids):
        score_row = query_vectors[row_index] @ doc_vectors.T
        score_vector = np.asarray(score_row.toarray()).ravel()
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": query_id,
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_ids) - 1:
            print(f"  [TF-IDF] processed {row_index + 1:,}/{len(query_ids):,} queries")
    return results


def run_bm25_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run lexical retrieval with BM25+ over tokenized text."""
    artifacts = prepared_artifacts or build_or_load_bm25_index(docs_frame)
    bm25 = artifacts["bm25"]
    doc_ids = artifacts["doc_ids"]
    capped_top_k = min(top_k, len(doc_ids))
    query_pairs = list(queries_frame[["id", "content"]].itertuples(index=False, name=None))
    log_every = progress_interval(len(query_pairs))

    print(
        f"  [BM25+] scoring {len(query_pairs):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, (query_id, query_text) in enumerate(query_pairs):
        score_vector = np.asarray(bm25.get_scores(tokenize(query_text)), dtype=np.float32)
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": str(query_id),
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_pairs) - 1:
            print(f"  [BM25+] processed {row_index + 1:,}/{len(query_pairs):,} queries")
    return results


def run_embedding_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Run dense semantic retrieval with Sentence-Transformer embeddings."""
    artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame)
    model = artifacts["model"]
    doc_embeddings = artifacts["doc_embeddings"]
    doc_ids = artifacts["doc_ids"]
    query_embeddings = _load_or_encode_embeddings(
        queries_frame,
        kind=embedding_kind,
        model=model,
        model_name=EMBEDDING_MODEL_NAME,
        batch_size=EMBEDDING_BATCH_SIZE,
    )

    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    results: list[RetrievalResult] = []
    total_chunks = (len(query_embeddings) + EMBEDDING_QUERY_CHUNK_SIZE - 1) // EMBEDDING_QUERY_CHUNK_SIZE

    print(
        f"  [Embedding] scoring {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}, chunk_size={EMBEDDING_QUERY_CHUNK_SIZE:,}, "
        f"embedding_cache_key='{embedding_kind}'"
    )

    for chunk_index, start_index in enumerate(range(0, len(query_embeddings), EMBEDDING_QUERY_CHUNK_SIZE), start=1):
        stop_index = start_index + EMBEDDING_QUERY_CHUNK_SIZE
        print(
            f"  [Embedding] chunk {chunk_index:,}/{total_chunks:,}: "
            f"queries {start_index + 1:,}-{min(stop_index, len(query_embeddings)):,}"
        )
        score_block = query_embeddings[start_index:stop_index] @ doc_embeddings.T
        for row_offset, score_vector in enumerate(score_block):
            top_indices = top_k_indices(score_vector, capped_top_k)
            query_id = query_ids[start_index + row_offset]
            results.append(
                {
                    "query_id": query_id,
                    "relevant_docs": doc_ids[top_indices].tolist(),
                }
            )
    return results


MODELS: dict[ModelName, Any] = {
    "tfidf": run_tfidf_search,
    "bm25": run_bm25_search,
    "embedding": run_embedding_search,
}


def run_retrieval(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Dispatch retrieval to the selected method and log the runtime."""
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(MODELS)}")

    print("=" * 88)
    print(f"Starting retrieval: model={model_name}")
    print(
        f"  parameters: top_k={top_k:,}, docs={len(docs_frame):,}, queries={len(queries_frame):,}, "
        f"prepared_artifacts={'yes' if prepared_artifacts is not None else 'no'}, "
        f"embedding_kind='{embedding_kind}'"
    )
    start_time = time.time()
    if model_name == "embedding":
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
            embedding_kind=embedding_kind,
        )
    else:
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
        )
    elapsed_seconds = time.time() - start_time
    print(
        f"Completed retrieval: model={model_name}, results={len(results):,} queries, "
        f"elapsed={elapsed_seconds:.1f}s"
    )
    print("=" * 88)
    return results
validate_pipeline_settings(document_count=len(docs_df))
RETRIEVAL_TOP_K = SUBMIT_TOP_K
print(f"Submission top_k : {SUBMIT_TOP_K:,}")
print(f"Retrieval top_k  : {RETRIEVAL_TOP_K:,}")


## Evaluation Logic

Offline evaluation mirrors the competition-style retrieval objective. The notebook reports:
- **Recall@K**: how much of the relevant set is recovered
- **Precision@K**: how much of the returned list is relevant
- **MRR@K**: how early the first relevant document appears
- **Accuracy**: category prediction accuracy from the lightweight query classifier

The combined offline score is the simple average of those four values.


In [ ]:
def load_ground_truth(path: Path) -> dict[str, GroundTruthEntry]:
    """Load the training relevance annotations from `qgts_train.json`."""
    if not path.exists():
        raise FileNotFoundError(f"Ground-truth file not found: {path}")

    with open(path, "r", encoding="utf-8") as handle:
        raw_ground_truth = json.load(handle)

    ground_truth: dict[str, GroundTruthEntry] = {}
    for query_id, info in raw_ground_truth.items():
        relevant_items = info.get("relevant_doc_ids", [])
        ground_truth[str(query_id)] = {
            "relevant_doc_ids": {str(item["doc_id"]) for item in relevant_items},
            "total_relevant_docs": int(info.get("total_relevant_docs", len(relevant_items))),
            "category": info.get("category"),
        }
    return ground_truth


def recall_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Recall@K across all queries present in the ground truth."""
    recalls: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        total_relevant_docs = ground_truth[query_id]["total_relevant_docs"]
        predicted_doc_ids = item["relevant_docs"][:k]
        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        recall_value = hits / total_relevant_docs if total_relevant_docs > 0 else 0.0
        recalls.append(recall_value)

    return float(np.mean(recalls)) if recalls else 0.0


def precision_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Precision@K across all queries present in the ground truth."""
    precisions: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        predicted_doc_ids = item["relevant_docs"][:k]
        if not predicted_doc_ids:
            precisions.append(0.0)
            continue

        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        precisions.append(hits / len(predicted_doc_ids))

    return float(np.mean(precisions)) if precisions else 0.0


def mrr_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean reciprocal rank at K."""
    reciprocal_ranks: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(item["relevant_docs"][:k], start=1):
            if doc_id in relevant_doc_ids:
                reciprocal_rank = 1.0 / rank
                break
        reciprocal_ranks.append(reciprocal_rank)

    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0


def compute_category_accuracy(
    ground_truth: dict[str, GroundTruthEntry],
    predicted_categories: dict[str, str] | None,
    default_if_missing: float = 0.0,
) -> float:
    """Compute query category accuracy when category predictions are available."""
    if predicted_categories is None:
        return float(default_if_missing)

    try:
        from sklearn.metrics import accuracy_score
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    y_true: list[str] = []
    y_pred: list[str] = []
    for query_id, info in ground_truth.items():
        true_category = info.get("category")
        predicted_category = predicted_categories.get(str(query_id))
        if true_category is None or predicted_category is None:
            continue
        y_true.append(str(true_category))
        y_pred.append(str(predicted_category))

    if not y_true:
        return float(default_if_missing)
    return float(accuracy_score(y_true, y_pred))


def leaderboard_score(
    results: list[RetrievalResult],
    ground_truth: dict[str, GroundTruthEntry],
    k: int,
    predicted_categories: dict[str, str] | None = None,
    accuracy_value: float | None = None,
) -> dict[str, float]:
    """Compute the combined offline leaderboard-style score."""
    recall_value = recall_at_k(results, ground_truth, k=k)
    precision_value = precision_at_k(results, ground_truth, k=k)
    mrr_value = mrr_at_k(results, ground_truth, k=k)
    category_accuracy = (
        float(accuracy_value)
        if accuracy_value is not None
        else compute_category_accuracy(ground_truth, predicted_categories)
    )
    combined_score = 0.25 * (recall_value + precision_value + mrr_value + category_accuracy)
    return {
        "Recall": recall_value,
        "Precision": precision_value,
        "MRR": mrr_value,
        "Accuracy": category_accuracy,
        "LeaderboardScore": combined_score,
    }


def predict_category_map(query_frame: pd.DataFrame, classifier_artifacts: dict[str, Any]) -> dict[str, str]:
    """Predict one category label per query using the cached classifier."""
    query_vectors = classifier_artifacts["vectorizer"].transform(query_frame["content"])
    predictions = classifier_artifacts["classifier"].predict(query_vectors)
    return {
        str(query_id): str(prediction)
        for query_id, prediction in zip(query_frame["id"].astype(str), predictions)
    }


def build_doc_category_map(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build a lookup from document id to document category."""
    require_columns(docs_frame, ["id", "category"], "Documents frame")
    return (
        docs_frame[["id", "category"]]
        .assign(id=lambda frame: frame["id"].astype(str))
        .set_index("id")["category"]
        .to_dict()
    )


def filter_results_by_dominant_category(
    results: list[RetrievalResult],
    doc_category_map: dict[str, Any],
    dominant_top_n: int = DOMINANT_CATEGORY_TOP_N,
) -> list[RetrievalResult]:
    """Keep only docs from the dominant category in each query's top-N window."""
    if dominant_top_n <= 0:
        raise ValueError("dominant_top_n must be positive.")

    filtered_results: list[RetrievalResult] = []
    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]
        top_window = doc_ids[:dominant_top_n]
        if not top_window:
            filtered_results.append({"query_id": query_id, "relevant_docs": []})
            continue

        category_counts: dict[str, int] = {}
        first_position: dict[str, int] = {}
        for position, doc_id in enumerate(top_window):
            raw_category = doc_category_map.get(doc_id)
            category = "unknown" if raw_category is None or pd.isna(raw_category) else str(raw_category)
            category_counts[category] = category_counts.get(category, 0) + 1
            if category not in first_position:
                first_position[category] = position

        dominant_category = min(
            category_counts,
            key=lambda category: (-category_counts[category], first_position[category], category),
        )
        filtered_docs = []
        for doc_id in doc_ids:
            raw_category = doc_category_map.get(doc_id)
            category = "unknown" if raw_category is None or pd.isna(raw_category) else str(raw_category)
            if category == dominant_category:
                filtered_docs.append(doc_id)
        filtered_results.append({"query_id": query_id, "relevant_docs": filtered_docs})
    return filtered_results


def filter_results_by_predicted_category(
    results: list[RetrievalResult],
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    keep_if_query_missing: bool = True,
) -> list[RetrievalResult]:
    """Keep only docs that match the classifier-predicted query category."""
    filtered_results: list[RetrievalResult] = []
    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]
        predicted_category = query_category_map.get(query_id)
        if predicted_category is None:
            retained_doc_ids = doc_ids if keep_if_query_missing else []
            filtered_results.append({"query_id": query_id, "relevant_docs": retained_doc_ids})
            continue

        target_category = str(predicted_category)
        filtered_doc_ids: list[str] = []
        for doc_id in doc_ids:
            raw_category = doc_category_map.get(doc_id)
            doc_category = "unknown" if raw_category is None or pd.isna(raw_category) else str(raw_category)
            if doc_category == target_category:
                filtered_doc_ids.append(doc_id)
        filtered_results.append({"query_id": query_id, "relevant_docs": filtered_doc_ids})
    return filtered_results


def build_text_map(frame: pd.DataFrame, id_column: str = "id", text_column: str = "content") -> dict[str, str]:
    """Build a string-id to text lookup from a dataframe."""
    require_columns(frame, [id_column, text_column], f"Frame[{id_column}, {text_column}]")
    return {
        str(row_id): str(text)
        for row_id, text in frame[[id_column, text_column]].itertuples(index=False, name=None)
    }


def sample_random_negative_doc_ids(
    all_doc_ids: Sequence[str],
    excluded_doc_ids: set[str],
    sample_size: int,
    rng: Any,
) -> list[str]:
    """Sample random negative doc ids while excluding known relevant ids."""
    if sample_size <= 0:
        return []
    if not all_doc_ids:
        return []

    negatives: list[str] = []
    max_attempts = max(100, sample_size * 50)
    attempts = 0
    while len(negatives) < sample_size and attempts < max_attempts:
        candidate = str(all_doc_ids[rng.randrange(len(all_doc_ids))])
        attempts += 1
        if candidate in excluded_doc_ids or candidate in negatives:
            continue
        negatives.append(candidate)
    return negatives


def mine_hard_negative_doc_ids(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    top_k: int = CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
    prepared_artifacts: dict[str, Any] | None = None,
) -> dict[str, list[str]]:
    """Mine hard negatives from top embedding hits that are not relevant."""
    if top_k <= 0:
        return {}

    query_id_set = {str(query_id) for query_id in query_ids}
    mining_queries_frame = (
        train_queries_frame.assign(id=train_queries_frame["id"].astype(str))
        .loc[lambda frame: frame["id"].isin(query_id_set), ["id", "content"]]
        .reset_index(drop=True)
    )
    if mining_queries_frame.empty:
        return {}

    embedding_artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame)
    retrieval_results = run_retrieval(
        model_name="embedding",
        docs_frame=docs_frame,
        queries_frame=mining_queries_frame,
        top_k=top_k,
        prepared_artifacts=embedding_artifacts,
        embedding_kind="queries_train_hardneg",
    )

    hard_negative_doc_ids_by_query: dict[str, list[str]] = {}
    for item in retrieval_results:
        query_id = str(item["query_id"])
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"] if query_id in ground_truth else set()
        negatives: list[str] = []
        seen_doc_ids: set[str] = set()
        for doc_id in item["relevant_docs"]:
            candidate_doc_id = str(doc_id)
            if candidate_doc_id in relevant_doc_ids or candidate_doc_id in seen_doc_ids:
                continue
            negatives.append(candidate_doc_id)
            seen_doc_ids.add(candidate_doc_id)
        hard_negative_doc_ids_by_query[query_id] = negatives

    counts = [len(doc_ids) for doc_ids in hard_negative_doc_ids_by_query.values()]
    if counts:
        print(
            f"  [HardNegatives] mined for {len(counts):,} queries "
            f"(per-query min/mean/max={min(counts):,}/{float(np.mean(counts)):.1f}/{max(counts):,}, top_k={top_k:,})"
        )
    return hard_negative_doc_ids_by_query


def build_cross_encoder_training_examples(
    query_text_map: dict[str, str],
    doc_text_map: dict[str, str],
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    max_positives_per_query: int = CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
    negatives_per_positive: int = CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
    hard_negative_doc_ids_by_query: dict[str, list[str]] | None = None,
    seed: int = CROSS_ENCODER_RANDOM_SEED,
) -> list[Any]:
    """Build binary (query, doc) examples using positives plus mined hard negatives."""
    try:
        from sentence_transformers import InputExample
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    import random

    rng = random.Random(seed)
    all_doc_ids = list(doc_text_map.keys())
    examples: list[Any] = []
    hard_negative_count = 0
    random_negative_count = 0

    for query_id in query_ids:
        query_id_str = str(query_id)
        query_text = query_text_map.get(query_id_str)
        if query_text is None:
            continue
        if query_id_str not in ground_truth:
            continue

        relevant_doc_ids = [
            str(doc_id)
            for doc_id in ground_truth[query_id_str]["relevant_doc_ids"]
            if str(doc_id) in doc_text_map
        ]
        if not relevant_doc_ids:
            continue

        rng.shuffle(relevant_doc_ids)
        selected_positive_doc_ids = relevant_doc_ids[:max_positives_per_query]
        relevant_doc_id_set = set(relevant_doc_ids)
        hard_negative_pool = [
            doc_id
            for doc_id in (hard_negative_doc_ids_by_query or {}).get(query_id_str, [])
            if doc_id in doc_text_map and doc_id not in relevant_doc_id_set
        ]

        for positive_doc_id in selected_positive_doc_ids:
            examples.append(InputExample(texts=[query_text, doc_text_map[positive_doc_id]], label=1.0))
            negative_doc_ids: list[str] = []

            if hard_negative_pool:
                hard_take = min(negatives_per_positive, len(hard_negative_pool))
                negative_doc_ids.extend(rng.sample(hard_negative_pool, hard_take))
                hard_negative_count += hard_take

            if len(negative_doc_ids) < negatives_per_positive:
                random_needed = negatives_per_positive - len(negative_doc_ids)
                extra_random_negatives = sample_random_negative_doc_ids(
                    all_doc_ids=all_doc_ids,
                    excluded_doc_ids=relevant_doc_id_set | set(negative_doc_ids),
                    sample_size=random_needed,
                    rng=rng,
                )
                negative_doc_ids.extend(extra_random_negatives)
                random_negative_count += len(extra_random_negatives)

            for negative_doc_id in negative_doc_ids:
                examples.append(InputExample(texts=[query_text, doc_text_map[negative_doc_id]], label=0.0))

    print(
        f"Cross-encoder pairs: total={len(examples):,}, "
        f"hard_negatives={hard_negative_count:,}, random_negatives={random_negative_count:,}"
    )
    return examples


def build_or_load_cross_encoder(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
) -> Any:
    """Train or load a cached cross-encoder reranker trained on ground-truth pairs."""
    try:
        from sentence_transformers import CrossEncoder
        from torch.utils.data import DataLoader
        import torch
    except ImportError as exc:
        raise ImportError(
            "Missing dependencies for cross-encoder training. Install `%pip install sentence-transformers torch`."
        ) from exc

    query_text_map = build_text_map(train_queries_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")

    candidate_query_ids = [
        query_id
        for query_id in train_queries_frame["id"].astype(str).tolist()
        if query_id in ground_truth and query_id in query_text_map
    ]
    if CROSS_ENCODER_TRAIN_QUERY_LIMIT > 0:
        candidate_query_ids = candidate_query_ids[:CROSS_ENCODER_TRAIN_QUERY_LIMIT]
    if not candidate_query_ids:
        raise ValueError("No training queries available for cross-encoder training.")

    train_signature = _hash_payload(
        {
            "query_signature": _dataframe_fingerprint(train_queries_frame, ["id", "content"]),
            "doc_signature": _dataframe_fingerprint(docs_frame, ["id", "content"]),
            "query_count": len(candidate_query_ids),
            "model": CROSS_ENCODER_MODEL_NAME,
            "epochs": CROSS_ENCODER_EPOCHS,
            "batch_size": CROSS_ENCODER_BATCH_SIZE,
            "max_length": CROSS_ENCODER_MAX_LENGTH,
            "max_pos_per_query": CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
            "neg_per_pos": CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
            "hard_neg_top_k": CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
            "hard_neg_model": "embedding",
            "hard_neg_embedding_model": EMBEDDING_MODEL_NAME,
            "seed": CROSS_ENCODER_RANDOM_SEED,
        }
    )
    model_dir = CROSS_ENCODER_CACHE_DIR / f"{_safe_component(CROSS_ENCODER_MODEL_NAME)}_{train_signature}"
    cache_marker = model_dir / "config.json"
    memory_key = f"cross_encoder::{model_dir.resolve()}"

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]

    if ENABLE_CROSS_ENCODER_CACHE and cache_marker.exists():
        print(f"Loading cross-encoder from cache: {model_dir.name}")
        try:
            cached_model = CrossEncoder(str(model_dir), max_length=CROSS_ENCODER_MAX_LENGTH)
            if CROSS_ENCODER_FP16 and torch.cuda.is_available():
                cached_model.model.half()
            _OBJECT_MEMORY_CACHE[memory_key] = cached_model
            return cached_model
        except Exception as exc:
            print(f"Cross-encoder cache load failed, retraining: {exc}")
    elif ENABLE_CROSS_ENCODER_CACHE and model_dir.exists():
        print(f"Cross-encoder cache directory exists but is incomplete: {model_dir}")

    embedding_artifacts = prepare_retriever("embedding", docs_frame)
    hard_negative_doc_ids_by_query = mine_hard_negative_doc_ids(
        train_queries_frame=train_queries_frame,
        docs_frame=docs_frame,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        top_k=CROSS_ENCODER_HARD_NEGATIVE_TOP_K,
        prepared_artifacts=embedding_artifacts,
    )

    training_examples = build_cross_encoder_training_examples(
        query_text_map=query_text_map,
        doc_text_map=doc_text_map,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        max_positives_per_query=CROSS_ENCODER_MAX_POSITIVES_PER_QUERY,
        negatives_per_positive=CROSS_ENCODER_NEGATIVES_PER_POSITIVE,
        hard_negative_doc_ids_by_query=hard_negative_doc_ids_by_query,
        seed=CROSS_ENCODER_RANDOM_SEED,
    )
    if not training_examples:
        raise ValueError("Cross-encoder training set is empty after preprocessing.")

    print(
        f"Training cross-encoder on {len(training_examples):,} pairs "
        f"from {len(candidate_query_ids):,} queries"
    )
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=CROSS_ENCODER_MAX_LENGTH)
    train_loader = DataLoader(training_examples, shuffle=True, batch_size=CROSS_ENCODER_BATCH_SIZE)
    warmup_steps = max(1, int(len(train_loader) * CROSS_ENCODER_EPOCHS * 0.1))
    if ENABLE_CROSS_ENCODER_CACHE:
        model_dir.mkdir(parents=True, exist_ok=True)
    output_path = str(model_dir) if ENABLE_CROSS_ENCODER_CACHE else None
    cross_encoder.fit(
        train_dataloader=train_loader,
        epochs=CROSS_ENCODER_EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        output_path=output_path,
    )

    if ENABLE_CROSS_ENCODER_CACHE:
        print(f"Saved cross-encoder to cache: {model_dir.name}")
        cross_encoder.save(str(model_dir))
        cached_model = CrossEncoder(str(model_dir), max_length=CROSS_ENCODER_MAX_LENGTH)
        if CROSS_ENCODER_FP16 and torch.cuda.is_available():
            cached_model.model.half()
        _OBJECT_MEMORY_CACHE[memory_key] = cached_model
        return cached_model

    if CROSS_ENCODER_FP16 and torch.cuda.is_available():
        cross_encoder.model.half()
    _OBJECT_MEMORY_CACHE[memory_key] = cross_encoder
    return cross_encoder
def compute_category_boost(
    query_id: str,
    doc_ids: list[str],
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    category_bonus: float
) -> np.ndarray:
    """
    Computes a flat scalar bonus for documents matching the predicted query category.
    This replaces the strict filter, preserving true positives that the classifier missed.
    """
    boosts = np.zeros(len(doc_ids), dtype=np.float32)
    predicted_query_cat = query_category_map.get(query_id)

    if predicted_query_cat is None or category_bonus == 0.0:
        return boosts

    target_category = str(predicted_query_cat)

    for i, doc_id in enumerate(doc_ids):
        raw_doc_cat = doc_category_map.get(doc_id)
        doc_category = "unknown" if raw_doc_cat is None or pd.isna(raw_doc_cat) else str(raw_doc_cat)
        
        if doc_category == target_category:
            boosts[i] = category_bonus

    return boosts

from sklearn.metrics import roc_auc_score, average_precision_score
def evaluate_isolated_cross_encoder(
    cross_encoder: Any,
    query_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, dict],
    hard_negative_doc_ids_by_query: dict[str, list[str]],
    sample_limit: int = 100
) -> None:
    """
    Evaluates the Cross-Encoder in isolation using standard classification metrics.
    Tests if the model can successfully score True Positives higher than Hard Negatives.
    """
    print("Running isolated Cross-Encoder diagnostics...")
    
    query_text_map = build_text_map(query_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    
    model_inputs = []
    labels = []
    
    queries_tested = 0
    for query_id, info in ground_truth.items():
        if queries_tested >= sample_limit:
            break
            
        query_id_str = str(query_id)
        if query_id_str not in query_text_map:
            continue
            
        query_text = query_text_map[query_id_str]
        relevant_doc_ids = [str(d) for d in info.get("relevant_doc_ids", []) if str(d) in doc_text_map]
        
        if not relevant_doc_ids:
            continue
            
        # Get up to 10 hard negatives for this query
        hard_negatives = hard_negative_doc_ids_by_query.get(query_id_str, [])
        hard_negatives = [d for d in hard_negatives if d in doc_text_map and d not in relevant_doc_ids][:10]
        
        if not hard_negatives:
            continue
            
        # Add the True Positives (Label 1)
        for doc_id in relevant_doc_ids:
            model_inputs.append([query_text, doc_text_map[doc_id]])
            labels.append(1.0)
            
        # Add the Hard Negatives (Label 0)
        for doc_id in hard_negatives:
            model_inputs.append([query_text, doc_text_map[doc_id]])
            labels.append(0.0)
            
        queries_tested += 1

    if not labels:
        print("Not enough valid pairs to run diagnostics.")
        return

    # Run the isolated predictions
    print(f"Scoring {len(labels)} pairs ({sum(labels)} Positives, {len(labels)-sum(labels)} Negatives)...")
    scores = np.asarray(
        cross_encoder.predict(model_inputs, batch_size=32, show_progress_bar=True),
        dtype=np.float32
    ).reshape(-1)
    
    # Calculate Metrics
    roc_auc = roc_auc_score(labels, scores)
    pr_auc = average_precision_score(labels, scores)
    
    print("\n=== Isolated Cross-Encoder Metrics ===")
    print(f"ROC-AUC:           {roc_auc:.4f} (1.0 is perfect separation, 0.5 is random guessing)")
    print(f"Average Precision: {pr_auc:.4f} (Higher is better, measures precision-recall curve)")
    
    # Sanity check: average scores
    pos_scores = scores[np.array(labels) == 1.0]
    neg_scores = scores[np.array(labels) == 0.0]
    print(f"\nMean Score (Positives): {np.mean(pos_scores):.4f}")
    print(f"Mean Score (Negatives): {np.mean(neg_scores):.4f}")

def rerank_results_with_cross_encoder(
    results: list[RetrievalResult],
    query_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    cross_encoder: Any,
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    rerank_top_m: int = 100,  # Adjusted to 100 per your request
    category_bonus: float = 2.0,
    return_diagnostics: bool = False,
) -> list[RetrievalResult] | tuple[list[RetrievalResult], list[dict[str, Any]]]:
    """
    Reranks candidates purely by raw Cross-Encoder scores + a soft category bonus.
    Completely ignores the base retriever's ranking for the top_m documents.

    When `return_diagnostics=True`, also returns per-query score traces so evaluation
    can explain where reranking helped or hurt.
    """
    if rerank_top_m <= 0:
        raise ValueError("rerank_top_m must be positive.")

    query_text_map = build_text_map(query_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    # --- Pass 1: collect all pairs and per-query metadata ---
    all_pairs: list[list[str]] = []
    query_meta: list[dict[str, Any]] = []

    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]

        if query_id not in query_text_map:
            query_meta.append(
                {
                    "query_id": query_id,
                    "skip": True,
                    "skip_reason": "missing_query_text",
                    "doc_ids": doc_ids,
                    "head_doc_ids": doc_ids[:rerank_top_m],
                }
            )
            continue

        head_doc_ids = doc_ids[:rerank_top_m]
        tail_doc_ids = doc_ids[rerank_top_m:]
        scored_doc_ids = [d for d in head_doc_ids if d in doc_text_map]
        missing_head_doc_ids = [d for d in head_doc_ids if d not in doc_text_map]

        if len(scored_doc_ids) <= 1:
            query_meta.append(
                {
                    "query_id": query_id,
                    "skip": True,
                    "skip_reason": "insufficient_scored_docs",
                    "doc_ids": doc_ids,
                    "head_doc_ids": head_doc_ids,
                }
            )
            continue

        query_text = query_text_map[query_id]
        pairs = [[query_text, doc_text_map[d]] for d in scored_doc_ids]
        start = len(all_pairs)
        all_pairs.extend(pairs)
        query_meta.append(
            {
                "query_id": query_id,
                "skip": False,
                "head_doc_ids": head_doc_ids,
                "scored_doc_ids": scored_doc_ids,
                "missing_head_doc_ids": missing_head_doc_ids,
                "tail_doc_ids": tail_doc_ids,
                "slice": (start, start + len(pairs)),
            }
        )

    # --- Single global predict() call across all queries ---
    if all_pairs:
        all_scores = np.asarray(
            cross_encoder.predict(
                all_pairs,
                batch_size=CROSS_ENCODER_INFER_BATCH_SIZE,
                show_progress_bar=False,
            ),
            dtype=np.float32,
        )
    else:
        all_scores = np.array([], dtype=np.float32)

    # --- Pass 2: reconstruct reranked results ---
    reranked_results: list[RetrievalResult] = []
    rerank_diagnostics: list[dict[str, Any]] = []
    reranked_query_count = 0

    for meta in query_meta:
        if meta["skip"]:
            reranked_results.append({"query_id": meta["query_id"], "relevant_docs": meta["doc_ids"]})
            if return_diagnostics:
                rerank_diagnostics.append(
                    {
                        "query_id": meta["query_id"],
                        "reranked": False,
                        "skip_reason": meta["skip_reason"],
                        "rerank_top_m": int(rerank_top_m),
                        "category_bonus": float(category_bonus),
                        "original_head_doc_ids": meta.get("head_doc_ids", []),
                        "reranked_head_doc_ids": meta.get("head_doc_ids", []),
                        "missing_head_doc_ids": [],
                        "tail_doc_ids_count": max(0, len(meta["doc_ids"]) - len(meta.get("head_doc_ids", []))),
                        "candidate_docs": [],
                    }
                )
            continue

        start, end = meta["slice"]
        ce_scores = all_scores[start:end]
        scored_doc_ids = meta["scored_doc_ids"]
        boosts = compute_category_boost(
            query_id=meta["query_id"],
            doc_ids=scored_doc_ids,
            query_category_map=query_category_map,
            doc_category_map=doc_category_map,
            category_bonus=category_bonus,
        )
        final_scores = ce_scores + boosts
        ranked_indices = np.argsort(final_scores)[::-1]
        reranked_scored_doc_ids = [scored_doc_ids[i] for i in ranked_indices]
        reranked_doc_ids = reranked_scored_doc_ids + meta["missing_head_doc_ids"] + meta["tail_doc_ids"]
        reranked_results.append({"query_id": meta["query_id"], "relevant_docs": reranked_doc_ids})
        reranked_query_count += 1

        if return_diagnostics:
            reranked_positions = {
                doc_id: rank
                for rank, doc_id in enumerate(reranked_scored_doc_ids, start=1)
            }
            candidate_docs: list[dict[str, Any]] = []
            for original_rank, doc_id in enumerate(scored_doc_ids, start=1):
                raw_doc_category = doc_category_map.get(doc_id)
                doc_category = "unknown" if raw_doc_category is None or pd.isna(raw_doc_category) else str(raw_doc_category)
                score_index = original_rank - 1
                candidate_docs.append(
                    {
                        "doc_id": doc_id,
                        "original_rank": int(original_rank),
                        "reranked_rank": int(reranked_positions[doc_id]),
                        "cross_encoder_score": float(ce_scores[score_index]),
                        "category_boost": float(boosts[score_index]),
                        "final_score": float(final_scores[score_index]),
                        "doc_category": doc_category,
                    }
                )

            rerank_diagnostics.append(
                {
                    "query_id": meta["query_id"],
                    "reranked": True,
                    "skip_reason": None,
                    "rerank_top_m": int(rerank_top_m),
                    "category_bonus": float(category_bonus),
                    "predicted_query_category": query_category_map.get(meta["query_id"]),
                    "original_head_doc_ids": meta["head_doc_ids"],
                    "reranked_head_doc_ids": reranked_scored_doc_ids + meta["missing_head_doc_ids"],
                    "missing_head_doc_ids": meta["missing_head_doc_ids"],
                    "tail_doc_ids_count": len(meta["tail_doc_ids"]),
                    "candidate_docs": candidate_docs,
                }
            )

    print(
        f"  [CrossEncoder] reranked {reranked_query_count:,}/{len(results):,} queries "
        f"(top_m={rerank_top_m:,}, category_bonus={category_bonus:.2f}, "
        f"total_pairs={len(all_pairs):,})"
    )
    if return_diagnostics:
        return reranked_results, rerank_diagnostics
    return reranked_results


def write_kaggle_submission(
    results: list[RetrievalResult],
    sample_csv_path: Path,
    output_csv_path: Path,
    category_predictions: dict[str, str] | None = None,
) -> None:
    """Write predictions in the exact Kaggle submission format."""
    prediction_map = {
        str(item["query_id"]): [str(doc_id) for doc_id in item["relevant_docs"]]
        for item in results
    }

    with open(sample_csv_path, "r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)

    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError("Invalid sample submission format.")

    query_id_column = fieldnames[0]
    prediction_column = fieldnames[1]
    category_column = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            query_id = str(row[query_id_column])
            if query_id not in prediction_map:
                raise ValueError(f"Missing retrieval prediction for query_id={query_id}")

            output_row = {
                query_id_column: query_id,
                prediction_column: json.dumps(prediction_map[query_id]),
            }
            if category_column is not None:
                if category_predictions is None:
                    output_row[category_column] = row.get(category_column, "?") or "?"
                else:
                    if query_id not in category_predictions:
                        raise ValueError(f"Missing category prediction for query_id={query_id}")
                    output_row[category_column] = str(category_predictions[query_id])
            writer.writerow(output_row)


ground_truth = load_ground_truth(ground_truth_path)
print(f"Ground-truth queries: {len(ground_truth):,}")


## Interpreting Recall, Precision, MRR, and Accuracy

The four reported metrics answer different questions:

- **Recall@K**: “Did we recover most of the relevant documents at all?”
- **Precision@K**: “How much noise is in the returned list?”
- **MRR@K**: “How early does the first useful document appear?”
- **Accuracy**: “Did the classifier predict the correct category label?”

Interpretation tips:
- If recall is low, the first-stage retriever is failing to surface relevant candidates.
- If recall is high but MRR is low, the right documents are present but ranked too deep.
- If precision falls sharply as K grows, the extra tail is mostly noise.
- If accuracy is stable across runs while retrieval metrics move, the changes are coming from the retriever rather than the classifier.


## Practical Tradeoffs in the Pipeline

A few practical engineering lessons matter more than clever modeling here:

- **Cache expensive artifacts**. Dense embeddings and fitted indexes are too expensive to rebuild every run.
- **Keep preprocessing shared**. Small text-normalization differences across methods make comparisons noisy and harder to trust.
- **Optimize the bottlenecks first**. In this notebook the biggest wins come from avoiding repeated scoring work and full score sorting.
- **Improve the first-stage retriever first**. If the recall ceiling is weak, later refinements will not rescue the pipeline.
- **Use lexical methods intentionally**. They remain useful for exact strings, error codes, product names, and very tight latency budgets.


## Cross-Encoder Reranking

A cross-encoder is a **second-stage ranker**: instead of encoding query and document independently, it scores each `(query, document)` pair jointly.

Why this helps MRR:
- first-stage retrieval focuses on recall and usually returns a noisy candidate set
- cross-encoders are slower but better at fine-grained ordering near the top of the list
- better ordering at the top is exactly what MRR rewards

In this notebook we keep the expensive part bounded:
- train on ground-truth query-document pairs (positive + sampled negative examples)
- rerank only the **first 20 queries** during offline experiments
- rerank only the top segment (`top_m`) per query to control runtime


## Experiments

The experiment loop below keeps the current notebook behavior but makes the logic more explicit:
- train the category classifier once
- train or load the cross-encoder reranker from ground truth
- prepare retrieval artifacts once per model
- retrieve a **single max-k ranking** per model on the train queries
- rerank the first 20 queries with the cross-encoder
- reuse that ranking for smaller K values by truncation

Reusing the max-k ranking is a safe optimization because the top-`k_small` prefix of a correctly sorted top-`k_large` ranking is identical to rerunning retrieval directly at `k_small`.


In [ ]:
category_classifier = build_or_load_category_classifier(docs_classifier_df)
train_query_category_map = predict_category_map(train_queries_classifier_df, category_classifier)
test_query_category_map = predict_category_map(test_queries_classifier_df, category_classifier)
classifier_accuracy = compute_category_accuracy(ground_truth, train_query_category_map)

print(f"Classifier accuracy on train queries: {classifier_accuracy:.4f}")
print(f"Train category predictions         : {len(train_query_category_map):,}")
print(f"Test category predictions          : {len(test_query_category_map):,}")

TOP_CATEGORY_STATS_K = 20
QUERY_CATEGORY_STATS_LIMIT = 5
doc_category_map = build_doc_category_map(docs_df)
cross_encoder_reranker = None
if ENABLE_CROSS_ENCODER_RERANK:
    cross_encoder_reranker = build_or_load_cross_encoder(
        train_queries_frame=train_queries_df,
        docs_frame=docs_df,
        ground_truth=ground_truth,
    )

def format_category_percentages(doc_ids: list[str], doc_categories: dict[str, Any]) -> str:
    """Format category percentages for one query's retrieved document ids."""
    category_counts: dict[str, int] = {}
    total_docs = 0

    for doc_id in doc_ids:
        raw_category = doc_categories.get(str(doc_id))
        if raw_category is None or pd.isna(raw_category):
            category = "unknown"
        else:
            category = str(raw_category)
        category_counts[category] = category_counts.get(category, 0) + 1
        total_docs += 1

    if total_docs == 0:
        return ""

    sorted_counts = sorted(category_counts.items(), key=lambda item: (-item[1], item[0]))
    return " | ".join(
        f"{category}: {100.0 * count / total_docs:.2f}%"
        for category, count in sorted_counts
    )


def top_category_percentages_by_query(
    results: list[RetrievalResult],
    doc_categories: dict[str, Any],
    top_n: int = TOP_CATEGORY_STATS_K,
) -> dict[str, str]:
    """Return one category-percentage summary per query."""
    return {
        str(item["query_id"]): format_category_percentages(item["relevant_docs"][:top_n], doc_categories)
        for item in results
    }


def preview_text(value: Any, limit: int = 280) -> str:
    """Collapse whitespace and trim long text for JSON diagnostics."""
    if value is None or pd.isna(value):
        return ""
    text = re.sub(r"\s+", " ", str(value)).strip()
    if len(text) <= limit:
        return text
    return text[: max(0, limit - 3)].rstrip() + "..."


def build_scalar_map(frame: pd.DataFrame, value_column: str) -> dict[str, Any]:
    """Build an id-to-scalar lookup while normalizing missing values to None."""
    require_columns(frame, ["id", value_column], f"Frame[id, {value_column}]")
    return {
        str(row_id): (None if pd.isna(value) else str(value))
        for row_id, value in frame[["id", value_column]].itertuples(index=False, name=None)
    }


def to_json_ready(value: Any) -> Any:
    """Recursively convert notebook objects into JSON-safe Python values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): to_json_ready(subvalue) for key, subvalue in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_ready(item) for item in value]
    if isinstance(value, set):
        return [to_json_ready(item) for item in sorted(value)]
    if isinstance(value, np.ndarray):
        return [to_json_ready(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if pd.isna(value):
        return None
    return str(value)


def compute_query_metrics(
    result: RetrievalResult,
    ground_truth: dict[str, GroundTruthEntry],
    k: int,
) -> dict[str, Any]:
    """Compute detailed per-query retrieval metrics and keep full retrieved ids."""
    query_id = str(result["query_id"])
    if query_id not in ground_truth:
        raise KeyError(f"Missing ground truth for query_id={query_id}")

    relevant_doc_id_set = {str(doc_id) for doc_id in ground_truth[query_id]["relevant_doc_ids"]}
    relevant_doc_ids = sorted(relevant_doc_id_set)
    predicted_doc_ids = [str(doc_id) for doc_id in result["relevant_docs"][:k]]
    predicted_doc_id_set = set(predicted_doc_ids)
    retrieved_relevant_doc_ids = [doc_id for doc_id in predicted_doc_ids if doc_id in relevant_doc_id_set]
    hit_count = len(retrieved_relevant_doc_ids)
    total_relevant_docs = int(ground_truth[query_id]["total_relevant_docs"])
    first_relevant_rank = next(
        (rank for rank, doc_id in enumerate(predicted_doc_ids, start=1) if doc_id in relevant_doc_id_set),
        None,
    )
    reciprocal_rank = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    recall_value = hit_count / total_relevant_docs if total_relevant_docs > 0 else 0.0
    precision_value = hit_count / len(predicted_doc_ids) if predicted_doc_ids else 0.0

    return {
        "total_relevant_docs": total_relevant_docs,
        "relevant_doc_ids": relevant_doc_ids,
        "retrieved_doc_ids": predicted_doc_ids,
        "retrieved_relevant_doc_ids": retrieved_relevant_doc_ids,
        "missed_relevant_doc_ids": [doc_id for doc_id in relevant_doc_ids if doc_id not in predicted_doc_id_set],
        "hit_count": int(hit_count),
        "recall_at_k": float(recall_value),
        "precision_at_k": float(precision_value),
        "first_relevant_rank": None if first_relevant_rank is None else int(first_relevant_rank),
        "reciprocal_rank": float(reciprocal_rank),
    }


def build_relevant_rank_changes(
    base_doc_ids: list[str],
    reranked_doc_ids: list[str],
    relevant_doc_ids: list[str],
) -> list[dict[str, Any]]:
    """Track how each relevant document moved after reranking."""
    base_positions = {doc_id: rank for rank, doc_id in enumerate(base_doc_ids, start=1)}
    reranked_positions = {doc_id: rank for rank, doc_id in enumerate(reranked_doc_ids, start=1)}
    changes: list[dict[str, Any]] = []
    for doc_id in relevant_doc_ids:
        base_rank = base_positions.get(doc_id)
        reranked_rank = reranked_positions.get(doc_id)
        rank_gain = None
        if base_rank is not None and reranked_rank is not None:
            rank_gain = int(base_rank - reranked_rank)
        changes.append(
            {
                "doc_id": doc_id,
                "base_rank": None if base_rank is None else int(base_rank),
                "reranked_rank": None if reranked_rank is None else int(reranked_rank),
                "rank_gain": rank_gain,
            }
        )
    return changes


query_title_map = build_scalar_map(train_queries_raw_df, "title")
query_text_map = build_text_map(train_queries_df, id_column="id", text_column="content")
true_query_category_map = build_scalar_map(train_queries_raw_df, "category")
doc_title_map = build_scalar_map(docs_raw_df, "title")
doc_raw_text_map = build_scalar_map(docs_raw_df, "text")


def build_doc_snapshot(doc_id: str, rank: int | None, relevant_doc_ids: set[str]) -> dict[str, Any]:
    """Build a compact human-readable snapshot for bad-case analysis."""
    raw_doc_category = doc_category_map.get(str(doc_id))
    doc_category = None if raw_doc_category is None or pd.isna(raw_doc_category) else str(raw_doc_category)
    return {
        "rank": None if rank is None else int(rank),
        "doc_id": str(doc_id),
        "title": doc_title_map.get(str(doc_id)),
        "category": doc_category,
        "text_preview": preview_text(doc_raw_text_map.get(str(doc_id))),
        "is_relevant": str(doc_id) in relevant_doc_ids,
    }


def build_ranked_doc_debug_row(
    doc_id: str,
    relevant_doc_ids: set[str],
    candidate_doc_map: dict[str, dict[str, Any]],
    predicted_query_category: str | None,
    base_rank: int | None = None,
    reranked_rank: int | None = None,
) -> dict[str, Any]:
    """Build a readable debug row for one document in a reranker failure case."""
    raw_doc_category = doc_category_map.get(str(doc_id))
    doc_category = None if raw_doc_category is None or pd.isna(raw_doc_category) else str(raw_doc_category)
    matches_predicted_category = (
        None
        if predicted_query_category is None or doc_category is None
        else doc_category == str(predicted_query_category)
    )
    candidate_info = candidate_doc_map.get(str(doc_id), {})
    rank_gain = None
    if base_rank is not None and reranked_rank is not None:
        rank_gain = int(base_rank - reranked_rank)
    return {
        "title": doc_title_map.get(str(doc_id)),
        "text_preview": preview_text(doc_raw_text_map.get(str(doc_id))),
        "doc_category": doc_category,
        "matches_predicted_category": matches_predicted_category,
        "base_rank": None if base_rank is None else int(base_rank),
        "reranked_rank": None if reranked_rank is None else int(reranked_rank),
        "rank_gain": rank_gain,
        "cross_encoder_score": candidate_info.get("cross_encoder_score"),
        "category_boost": candidate_info.get("category_boost"),
        "final_score": candidate_info.get("final_score"),
        "is_relevant": str(doc_id) in relevant_doc_ids,
        "doc_id": str(doc_id),
    }


def summarize_reranker_failure(
    query_detail: dict[str, Any],
    demoted_relevant_rows: list[dict[str, Any]],
    promoted_non_relevant_rows: list[dict[str, Any]],
) -> list[str]:
    """Summarize the main ways the reranker hurt this query."""
    reasons: list[str] = []
    base_first_rank = query_detail.get("base_first_relevant_rank")
    reranked_first_rank = query_detail.get("first_relevant_rank")
    top_k = int(query_detail["top_k"])

    if base_first_rank is not None and reranked_first_rank is None:
        reasons.append(
            f"reranker removed the first relevant document from the top-{top_k} ranking"
        )
    elif (
        base_first_rank is not None
        and reranked_first_rank is not None
        and reranked_first_rank > base_first_rank
    ):
        reasons.append(
            f"first relevant moved from rank {base_first_rank} to rank {reranked_first_rank}"
        )

    if query_detail["delta_hit_count"] < 0:
        reasons.append(
            f"reranker dropped {abs(int(query_detail['delta_hit_count']))} relevant docs out of the top-{top_k} list"
        )

    if query_detail["category_correct"] is False:
        reasons.append(
            "query category prediction is wrong, so the category bonus may be pushing the ranking in the wrong direction"
        )

    if promoted_non_relevant_rows:
        top_false_positive = promoted_non_relevant_rows[0]
        doc_name = top_false_positive["title"] or top_false_positive["doc_id"]
        reasons.append(
            f"non-relevant document promoted by reranker: '{doc_name}'"
        )

    if demoted_relevant_rows:
        top_relevant = demoted_relevant_rows[0]
        doc_name = top_relevant["title"] or top_relevant["doc_id"]
        reasons.append(
            f"relevant document demoted by reranker: '{doc_name}'"
        )

    if not reasons:
        reasons.append("reranker regression detected, but no simple summary was extracted")
    return reasons


def build_bad_case_entry(query_detail: dict[str, Any], snapshot_limit: int = 10) -> dict[str, Any]:
    """Convert one reranker regression into a human-readable debug row."""
    query_id = str(query_detail["query_id"])
    relevant_doc_id_set = set(query_detail["relevant_doc_ids"])
    predicted_query_category = query_detail.get("predicted_category")
    rerank_diagnostics = query_detail.get("rerank_diagnostics") or {}
    candidate_doc_map = {
        str(item["doc_id"]): item
        for item in rerank_diagnostics.get("candidate_docs", [])
    }
    base_positions = {
        str(doc_id): rank
        for rank, doc_id in enumerate(query_detail["base_retrieved_doc_ids"], start=1)
    }
    reranked_positions = {
        str(doc_id): rank
        for rank, doc_id in enumerate(query_detail["retrieved_doc_ids"], start=1)
    }

    relevant_docs = [
        build_ranked_doc_debug_row(
            doc_id=doc_id,
            relevant_doc_ids=relevant_doc_id_set,
            candidate_doc_map=candidate_doc_map,
            predicted_query_category=predicted_query_category,
            base_rank=base_positions.get(str(doc_id)),
            reranked_rank=reranked_positions.get(str(doc_id)),
        )
        for doc_id in query_detail["relevant_doc_ids"]
    ]
    demoted_relevant_rows = sorted(
        [
            row
            for row in relevant_docs
            if row["base_rank"] is not None
            and (row["reranked_rank"] is None or row["reranked_rank"] > row["base_rank"])
        ],
        key=lambda row: (
            0 if row["reranked_rank"] is None else 1,
            -10**9 if row["reranked_rank"] is None else -(row["reranked_rank"] - row["base_rank"]),
            row["base_rank"] or 10**9,
        ),
    )

    promoted_non_relevant_rows = sorted(
        [
            build_ranked_doc_debug_row(
                doc_id=doc_id,
                relevant_doc_ids=relevant_doc_id_set,
                candidate_doc_map=candidate_doc_map,
                predicted_query_category=predicted_query_category,
                base_rank=base_positions.get(str(doc_id)),
                reranked_rank=reranked_positions.get(str(doc_id)),
            )
            for doc_id, candidate_info in candidate_doc_map.items()
            if doc_id not in relevant_doc_id_set
            and candidate_info.get("original_rank") is not None
            and candidate_info.get("reranked_rank") is not None
            and int(candidate_info["reranked_rank"]) < int(candidate_info["original_rank"])
        ],
        key=lambda row: (
            10**9 if row["base_rank"] is None or row["reranked_rank"] is None else -(row["base_rank"] - row["reranked_rank"]),
            row["reranked_rank"] or 10**9,
        ),
    )

    failure_reasons = summarize_reranker_failure(
        query_detail=query_detail,
        demoted_relevant_rows=demoted_relevant_rows,
        promoted_non_relevant_rows=promoted_non_relevant_rows,
    )

    return {
        "query": {
            "query_id": query_id,
            "title": query_title_map.get(query_id),
            "text_preview": preview_text(query_text_map.get(query_id)),
            "true_category": query_detail["true_category"],
            "predicted_category": query_detail["predicted_category"],
            "category_correct": query_detail["category_correct"],
        },
        "failure": {
            "delta_hit_count": query_detail["delta_hit_count"],
            "delta_recall_at_k": query_detail["delta_recall_at_k"],
            "delta_precision_at_k": query_detail["delta_precision_at_k"],
            "delta_reciprocal_rank": query_detail["delta_reciprocal_rank"],
            "base_first_relevant_rank": query_detail["base_first_relevant_rank"],
            "reranked_first_relevant_rank": query_detail["first_relevant_rank"],
            "base_hit_count": query_detail["base_hit_count"],
            "reranked_hit_count": query_detail["hit_count"],
            "failure_reasons": failure_reasons,
        },
        "reranker_context": {
            "reranked": rerank_diagnostics.get("reranked"),
            "rerank_top_m": rerank_diagnostics.get("rerank_top_m"),
            "category_bonus": rerank_diagnostics.get("category_bonus"),
            "predicted_query_category": rerank_diagnostics.get("predicted_query_category"),
            "candidate_doc_count": len(candidate_doc_map),
        },
        "relevant_docs": relevant_docs,
        "relevant_docs_hurt_by_reranker": demoted_relevant_rows[:snapshot_limit],
        "false_positives_promoted_by_reranker": promoted_non_relevant_rows[:snapshot_limit],
        "top_before_rerank": [
            build_ranked_doc_debug_row(
                doc_id=doc_id,
                relevant_doc_ids=relevant_doc_id_set,
                candidate_doc_map=candidate_doc_map,
                predicted_query_category=predicted_query_category,
                base_rank=rank,
                reranked_rank=reranked_positions.get(str(doc_id)),
            )
            for rank, doc_id in enumerate(query_detail["base_retrieved_doc_ids"][:snapshot_limit], start=1)
        ],
        "top_after_rerank": [
            build_ranked_doc_debug_row(
                doc_id=doc_id,
                relevant_doc_ids=relevant_doc_id_set,
                candidate_doc_map=candidate_doc_map,
                predicted_query_category=predicted_query_category,
                base_rank=base_positions.get(str(doc_id)),
                reranked_rank=rank,
            )
            for rank, doc_id in enumerate(query_detail["retrieved_doc_ids"][:snapshot_limit], start=1)
        ],
    }


evaluation_output_dir = PATHS.work_dir / "report" / "evaluation"
evaluation_output_dir.mkdir(parents=True, exist_ok=True)
obsolete_detailed_evaluation_json_path = evaluation_output_dir / "evaluation_detailed.json"
if obsolete_detailed_evaluation_json_path.exists():
    obsolete_detailed_evaluation_json_path.unlink()
bad_cases_study_json_path = evaluation_output_dir / "bad_cases_study.json"

prepared_retrievers: dict[ModelName, dict[str, Any]] = {}
for model_name in set(EVALUATION_MODELS) | {FINAL_MODEL}:
    print(f"Preparing artifacts for {model_name}...")
    prepared_retrievers[model_name] = prepare_retriever(model_name, docs_df)

max_eval_top_k = max(EVALUATION_TOP_KS)
evaluation_rows: list[dict[str, Any]] = []
evaluation_query_rows: list[dict[str, Any]] = []
bad_cases_study: list[dict[str, Any]] = []

for model_name in EVALUATION_MODELS:
    base_max_k_results = run_retrieval(
        model_name=model_name,
        docs_frame=docs_df,
        queries_frame=train_queries_df,
        top_k=max_eval_top_k,
        prepared_artifacts=prepared_retrievers[model_name],
        embedding_kind="queries_train",
    )

    max_k_results = base_max_k_results
    current_bonus = 0.0
    rerank_diagnostics_by_query: dict[str, dict[str, Any]] = {}
    if cross_encoder_reranker is not None:
        current_bonus = CROSS_ENCODER_CATEGORY_BONUS if ENABLE_CATEGORY_FILTER else 0.0
        max_k_results, rerank_diagnostics = rerank_results_with_cross_encoder(
            results=base_max_k_results,
            query_frame=train_queries_df,
            docs_frame=docs_df,
            cross_encoder=cross_encoder_reranker,
            query_category_map=train_query_category_map,
            doc_category_map=doc_category_map,
            rerank_top_m=CROSS_ENCODER_RERANK_TOP_M,
            category_bonus=current_bonus,
            return_diagnostics=True,
        )
        rerank_diagnostics_by_query = {
            str(item["query_id"]): item
            for item in rerank_diagnostics
        }

    for top_k in EVALUATION_TOP_KS:
        base_truncated_results = truncate_results(base_max_k_results, top_k)
        truncated_results = truncate_results(max_k_results, top_k)
        metrics = leaderboard_score(
            truncated_results,
            ground_truth,
            k=top_k,
            accuracy_value=classifier_accuracy,
        )
        limited_results = truncated_results[:QUERY_CATEGORY_STATS_LIMIT]
        top20_category_pct_by_query = top_category_percentages_by_query(
            limited_results,
            doc_categories=doc_category_map,
            top_n=TOP_CATEGORY_STATS_K,
        )
        evaluation_rows.append(
            {
                "Model": model_name,
                "TopK": top_k,
                **metrics,
            }
        )
        for query_id, top20_category_pct in top20_category_pct_by_query.items():
            evaluation_query_rows.append(
                {
                    "Model": model_name,
                    "TopK": top_k,
                    "QueryID": query_id,
                    "Top20CategoryPct": top20_category_pct,
                }
            )

        base_result_by_query = {
            str(item["query_id"]): item
            for item in base_truncated_results
        }
        experiment_query_details: list[dict[str, Any]] = []
        for item in truncated_results:
            query_id = str(item["query_id"])
            query_metrics = compute_query_metrics(item, ground_truth, k=top_k)
            base_metrics = compute_query_metrics(base_result_by_query[query_id], ground_truth, k=top_k)
            true_category = true_query_category_map.get(query_id)
            predicted_category = train_query_category_map.get(query_id)
            category_correct = (
                None
                if true_category is None or predicted_category is None
                else str(true_category) == str(predicted_category)
            )
            query_detail = {
                "query_id": query_id,
                "query_title": query_title_map.get(query_id),
                "query_text": query_text_map.get(query_id),
                "true_category": true_category,
                "predicted_category": predicted_category,
                "category_correct": category_correct,
                "model": model_name,
                "top_k": int(top_k),
                **query_metrics,
                "base_retrieved_doc_ids": base_metrics["retrieved_doc_ids"],
                "base_hit_count": base_metrics["hit_count"],
                "base_recall_at_k": base_metrics["recall_at_k"],
                "base_precision_at_k": base_metrics["precision_at_k"],
                "base_first_relevant_rank": base_metrics["first_relevant_rank"],
                "base_reciprocal_rank": base_metrics["reciprocal_rank"],
                "delta_hit_count": int(query_metrics["hit_count"] - base_metrics["hit_count"]),
                "delta_recall_at_k": float(query_metrics["recall_at_k"] - base_metrics["recall_at_k"]),
                "delta_precision_at_k": float(query_metrics["precision_at_k"] - base_metrics["precision_at_k"]),
                "delta_reciprocal_rank": float(query_metrics["reciprocal_rank"] - base_metrics["reciprocal_rank"]),
                "relevant_doc_position_changes": build_relevant_rank_changes(
                    base_metrics["retrieved_doc_ids"],
                    query_metrics["retrieved_doc_ids"],
                    query_metrics["relevant_doc_ids"],
                ),
                "rerank_diagnostics": rerank_diagnostics_by_query.get(query_id),
            }
            experiment_query_details.append(query_detail)

        reranker_regression_candidates = sorted(
            (
                detail
                for detail in experiment_query_details
                if detail["delta_reciprocal_rank"] < 0.0 or detail["delta_hit_count"] < 0
            ),
            key=lambda detail: (
                detail["delta_reciprocal_rank"],
                detail["delta_hit_count"],
                -detail["base_reciprocal_rank"],
                detail["query_id"],
            ),
        )
        reranker_regressions = [
            build_bad_case_entry(detail)
            for detail in reranker_regression_candidates
        ]
        bad_cases_study.append(
            {
                "model": model_name,
                "top_k": int(top_k),
                "summary_metrics": {
                    "Recall": float(metrics["Recall"]),
                    "Precision": float(metrics["Precision"]),
                    "MRR": float(metrics["MRR"]),
                    "Accuracy": float(metrics["Accuracy"]),
                    "LeaderboardScore": float(metrics["LeaderboardScore"]),
                },
                "reranker_enabled": bool(cross_encoder_reranker is not None),
                "rerank_top_m": int(CROSS_ENCODER_RERANK_TOP_M),
                "category_bonus": float(current_bonus),
                "reranker_failure_summary": {
                    "query_count": len(experiment_query_details),
                    "failure_count": len(reranker_regression_candidates),
                    "mrr_drop_count": sum(
                        1 for detail in reranker_regression_candidates if detail["delta_reciprocal_rank"] < 0.0
                    ),
                    "recall_drop_count": sum(
                        1 for detail in reranker_regression_candidates if detail["delta_hit_count"] < 0
                    ),
                    "wrong_category_prediction_count": sum(
                        1 for detail in reranker_regression_candidates if detail["category_correct"] is False
                    ),
                },
                "reranker_regressions": reranker_regressions,
            }
        )

        print(
            f"{model_name:10s} top_k={top_k:>6,}  "
            f"Recall={metrics['Recall']:.5f}  "
            f"Precision={metrics['Precision']:.5f}  "
            f"MRR={metrics['MRR']:.5f}  "
            f"Accuracy={metrics['Accuracy']:.5f}  "
            f"Score={metrics['LeaderboardScore']:.5f}  "
            f"Top{TOP_CATEGORY_STATS_K}CatsByQuery={len(top20_category_pct_by_query):,} "
            f"(first {QUERY_CATEGORY_STATS_LIMIT} queries)"
        )

eval_summary_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["LeaderboardScore", "TopK"], ascending=[False, False])
    .reset_index(drop=True)
)

if eval_summary_df.empty:
    raise ValueError("No evaluation rows were produced.")

if not evaluation_query_rows:
    raise ValueError("No per-query category rows were produced.")

eval_df = (
    pd.DataFrame(evaluation_query_rows)
    .merge(eval_summary_df, on=["Model", "TopK"], how="left")
    .sort_values(["LeaderboardScore", "TopK", "QueryID"], ascending=[False, False, True])
    .reset_index(drop=True)
)

bad_cases_payload = {
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "runtime_env": PATHS.runtime_env,
    "settings": {
        "final_model": FINAL_MODEL,
        "evaluation_models": list(EVALUATION_MODELS),
        "evaluation_top_ks": list(EVALUATION_TOP_KS),
        "submit_top_k": int(SUBMIT_TOP_K),
        "enable_category_filter": bool(ENABLE_CATEGORY_FILTER),
        "enable_cross_encoder_rerank": bool(ENABLE_CROSS_ENCODER_RERANK),
        "cross_encoder_model_name": CROSS_ENCODER_MODEL_NAME,
        "cross_encoder_rerank_top_m": int(CROSS_ENCODER_RERANK_TOP_M),
        "cross_encoder_infer_batch_size": int(CROSS_ENCODER_INFER_BATCH_SIZE),
        "embedding_model_name": EMBEDDING_MODEL_NAME,
    },
    "experiments": bad_cases_study,
}

with open(bad_cases_study_json_path, "w", encoding="utf-8") as handle:
    json.dump(to_json_ready(bad_cases_payload), handle, ensure_ascii=False, indent=2)

print(f"Saved bad cases study JSON to    : {bad_cases_study_json_path.resolve()}")

best_experiment = eval_summary_df.iloc[0]
print()
print("Best offline configuration:")
print(best_experiment.to_string())
eval_df


## Cross-Encoder Diagnostics

Three quick health checks for the cross-encoder reranker:

- **Score Distribution** — histograms of CE scores on positive (relevant) vs hard-negative pairs. Good: clear separation between the two distributions (positive mean >> negative mean, ROC-AUC > 0.85). Bad: overlapping distributions or a score gap near zero — the model cannot distinguish relevant from irrelevant.

- **MRR Before vs After Reranking** — compares MRR@200 of the raw embedding ranking, CE-only reranking, and CE + category bonus. Good: each bar is higher than the previous. Bad: CE reranking hurts MRR relative to the embedding baseline.

- **Category Bonus Sensitivity Sweep** — plots MRR@10 as the `category_bonus` scalar is varied from 0 to 5. Good: a clear peak that is stable over a range (flat plateau around the optimum). Bad: a very sharp peak (fragile to small changes) or MRR monotonically falling as bonus increases (bonus is harmful).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# ── shared config ─────────────────────────────────────────────────────────────
_DIAG_N_QUERIES = 80
_DIAG_TOP_M_SWEEP = 150
_DIAG_RETRIEVAL_TOP_K = 200
_DIAG_HARD_NEG_PER_QUERY = 10
_DIAG_BONUS_GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
_DIAG_PRODUCTION_BONUS = CROSS_ENCODER_CATEGORY_BONUS
_DIAG_REPORT_DIR = PATHS.work_dir / "report"
_DIAG_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ── resolve category maps (use globals if available, else build inline) ───────
_diag_query_cat_map: dict = (
    train_query_category_map
    if "train_query_category_map" in dir()
    else predict_category_map(train_queries_df, category_classifier)
)
_diag_doc_cat_map: dict = (
    doc_category_map
    if "doc_category_map" in dir()
    else build_doc_category_map(docs_df)
)

# ── grab the first N train queries that have ground-truth entries ─────────────
_diag_query_ids: list[str] = [
    str(qid) for qid in train_queries_df["id"].tolist()[:_DIAG_N_QUERIES]
]
_diag_query_ids = [qid for qid in _diag_query_ids if qid in ground_truth]

_diag_query_frame = train_queries_df[
    train_queries_df["id"].astype(str).isin(_diag_query_ids)
].reset_index(drop=True)

# ── ensure embedding retriever is ready ──────────────────────────────────────
if "embedding" not in prepared_retrievers:
    print("Building embedding retriever for diagnostics …")
    prepared_retrievers["embedding"] = prepare_retriever("embedding", docs_df)
_diag_artifacts = prepared_retrievers["embedding"]

# ── retrieve top-200 results once (reused across all 3 diagnostics) ───────────
print(f"Retrieving top-{_DIAG_RETRIEVAL_TOP_K} docs for {len(_diag_query_ids)} queries …")
_diag_base_results = run_retrieval(
    model_name="embedding",
    docs_frame=docs_df,
    queries_frame=_diag_query_frame,
    top_k=_DIAG_RETRIEVAL_TOP_K,
    prepared_artifacts=_diag_artifacts,
    embedding_kind="queries_train",
)
_diag_results_by_qid: dict = {str(r["query_id"]): r for r in _diag_base_results}
_diag_all_doc_ids = set(docs_df["id"].astype(str).tolist())

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 1 — Score Distribution
# ══════════════════════════════════════════════════════════════════════════════
print("Running Diagnostic 1: Score Distribution …")
_diag_pairs: list[tuple[str, str]] = []
_diag_labels: list[float] = []
_rng = np.random.default_rng(42)

for qid in _diag_query_ids:
    gt_entry = ground_truth[qid]
    pos_ids = {str(d) for d in gt_entry["relevant_doc_ids"]}
    base_result = _diag_results_by_qid.get(qid)
    if base_result is None:
        continue
    retrieved_ids = [str(d) for d in base_result["relevant_docs"]]

    for pos_id in pos_ids:
        _diag_pairs.append((qid, pos_id))
        _diag_labels.append(1.0)

    hard_negs = [d for d in retrieved_ids if d not in pos_ids][:_DIAG_HARD_NEG_PER_QUERY]
    if not hard_negs:
        neg_pool = list(_diag_all_doc_ids - pos_ids)
        hard_negs = [
            neg_pool[i]
            for i in _rng.choice(len(neg_pool), size=min(_DIAG_HARD_NEG_PER_QUERY, len(neg_pool)), replace=False)
        ]
    for neg_id in hard_negs:
        _diag_pairs.append((qid, neg_id))
        _diag_labels.append(0.0)

_diag_query_texts = build_text_map(_diag_query_frame)
_diag_doc_texts = build_text_map(docs_df)
_diag_ce_inputs = [
    (_diag_query_texts.get(qid, ""), _diag_doc_texts.get(did, ""))
    for qid, did in _diag_pairs
]
_diag_scores = cross_encoder_reranker.predict(
    _diag_ce_inputs, batch_size=CROSS_ENCODER_INFER_BATCH_SIZE, show_progress_bar=True
)

_diag_labels_arr = np.array(_diag_labels)
_diag_scores_arr = np.array(_diag_scores, dtype=float)
_diag_pos_scores = _diag_scores_arr[_diag_labels_arr == 1.0]
_diag_neg_scores = _diag_scores_arr[_diag_labels_arr == 0.0]
_diag_roc_auc = roc_auc_score(_diag_labels_arr, _diag_scores_arr)
_diag_avg_prec = average_precision_score(_diag_labels_arr, _diag_scores_arr)
_diag_mean_pos = float(_diag_pos_scores.mean())
_diag_mean_neg = float(_diag_neg_scores.mean())
_diag_score_gap = _diag_mean_pos - _diag_mean_neg

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 2 — MRR Before vs After Reranking
# ══════════════════════════════════════════════════════════════════════════════
print("Running Diagnostic 2: MRR Before vs After Reranking …")
_diag_mrr_before = mrr_at_k(_diag_base_results, ground_truth, k=_DIAG_RETRIEVAL_TOP_K)

_diag_reranked_no_bonus, _ = rerank_results_with_cross_encoder(
    results=_diag_base_results,
    query_frame=_diag_query_frame,
    docs_frame=docs_df,
    cross_encoder=cross_encoder_reranker,
    query_category_map=_diag_query_cat_map,
    doc_category_map=_diag_doc_cat_map,
    rerank_top_m=_DIAG_RETRIEVAL_TOP_K,
    category_bonus=0.0,
    return_diagnostics=True,
)
_diag_mrr_after_ce = mrr_at_k(_diag_reranked_no_bonus, ground_truth, k=_DIAG_RETRIEVAL_TOP_K)

_diag_reranked_cat, _ = rerank_results_with_cross_encoder(
    results=_diag_base_results,
    query_frame=_diag_query_frame,
    docs_frame=docs_df,
    cross_encoder=cross_encoder_reranker,
    query_category_map=_diag_query_cat_map,
    doc_category_map=_diag_doc_cat_map,
    rerank_top_m=_DIAG_RETRIEVAL_TOP_K,
    category_bonus=_DIAG_PRODUCTION_BONUS,
    return_diagnostics=True,
)
_diag_mrr_after_ce_cat = mrr_at_k(_diag_reranked_cat, ground_truth, k=_DIAG_RETRIEVAL_TOP_K)

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC 3 — Category Bonus Sensitivity Sweep
# ══════════════════════════════════════════════════════════════════════════════
print("Running Diagnostic 3: Category Bonus Sensitivity Sweep …")
_diag_truncated_150 = [
    {"query_id": r["query_id"], "relevant_docs": r["relevant_docs"][:_DIAG_TOP_M_SWEEP]}
    for r in _diag_base_results
]
_diag_sweep_mrr: list[float] = []
for _bonus in _DIAG_BONUS_GRID:
    _reranked, _ = rerank_results_with_cross_encoder(
        results=_diag_truncated_150,
        query_frame=_diag_query_frame,
        docs_frame=docs_df,
        cross_encoder=cross_encoder_reranker,
        query_category_map=_diag_query_cat_map,
        doc_category_map=_diag_doc_cat_map,
        rerank_top_m=_DIAG_TOP_M_SWEEP,
        category_bonus=_bonus,
        return_diagnostics=True,
    )
    _diag_sweep_mrr.append(mrr_at_k(_reranked, ground_truth, k=10))

_diag_best_idx = int(np.argmax(_diag_sweep_mrr))
_diag_best_bonus = _DIAG_BONUS_GRID[_diag_best_idx]
_diag_best_mrr10 = _diag_sweep_mrr[_diag_best_idx]

# ══════════════════════════════════════════════════════════════════════════════
# PLOT
# ══════════════════════════════════════════════════════════════════════════════
plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Cross-Encoder Diagnostics", fontsize=14, fontweight="bold")

# ── Subplot 1: Score Distribution ─────────────────────────────────────────────
ax1 = axes[0]
ax1.hist(_diag_pos_scores, bins=30, alpha=0.6, color="green", label="Positive (relevant)")
ax1.hist(_diag_neg_scores, bins=30, alpha=0.6, color="red", label="Negative (hard neg)")
ax1.axvline(_diag_mean_pos, color="green", linestyle="--", linewidth=1.5,
            label=f"Mean pos = {_diag_mean_pos:.3f}")
ax1.axvline(_diag_mean_neg, color="red", linestyle="--", linewidth=1.5,
            label=f"Mean neg = {_diag_mean_neg:.3f}")
ax1.set_title("Score Distribution")
ax1.set_xlabel("Cross-Encoder Score")
ax1.set_ylabel("Count")
ax1.legend(fontsize=8)
ax1.set_ylim(bottom=0)
diag1_text = (
    f"ROC-AUC: {_diag_roc_auc:.4f}  (good > 0.85)\n"
    f"Avg Precision: {_diag_avg_prec:.4f}  (good > 0.80)\n"
    f"Mean pos: {_diag_mean_pos:.4f}  |  Mean neg: {_diag_mean_neg:.4f}\n"
    f"Score gap: {_diag_score_gap:.4f}  (good > 1.0)"
)
ax1.text(
    0.02, 0.97, diag1_text,
    transform=ax1.transAxes, fontsize=7.5, verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

# ── Subplot 2: MRR Before vs After ───────────────────────────────────────────
ax2 = axes[1]
_mrr_values = [_diag_mrr_before, _diag_mrr_after_ce, _diag_mrr_after_ce_cat]
_mrr_labels = ["Embedding only", "CE rerank\n(no bonus)", "CE rerank\n+ category bonus"]
_mrr_colors = ["#888888", "#4477CC", "#EE8833"]
bars = ax2.barh(_mrr_labels, _mrr_values, color=_mrr_colors)
ax2.set_xlim(0, max(_mrr_values) * 1.25)
ax2.set_title(f"MRR@{_DIAG_RETRIEVAL_TOP_K}: Before vs After Reranking")
ax2.set_xlabel(f"MRR@{_DIAG_RETRIEVAL_TOP_K}")
ax2.set_ylabel("Ranker")
for bar, val in zip(bars, _mrr_values):
    ax2.text(
        bar.get_width() + max(_mrr_values) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}", va="center", ha="left", fontsize=9, fontweight="bold",
    )
ax2.set_ylim(bottom=-0.5)
_delta_ce = _diag_mrr_after_ce - _diag_mrr_before
_delta_cat = _diag_mrr_after_ce_cat - _diag_mrr_after_ce
ax2.text(
    0.02, 0.06,
    f"CE vs emb: {_delta_ce:+.4f}\nCE+cat vs CE: {_delta_cat:+.4f}",
    transform=ax2.transAxes, fontsize=8,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

# ── Subplot 3: Category Bonus Sensitivity ─────────────────────────────────────
ax3 = axes[2]
ax3.plot(_DIAG_BONUS_GRID, _diag_sweep_mrr, marker="o", color="#4477CC",
         linewidth=2, label="MRR@10")
ax3.axvline(_DIAG_PRODUCTION_BONUS, color="red", linestyle="--", linewidth=1.5,
            label=f"current ({_DIAG_PRODUCTION_BONUS})")
ax3.plot(
    _diag_best_bonus, _diag_best_mrr10,
    marker="*", color="gold", markersize=18, zorder=5,
    label=f"best={_diag_best_bonus} ({_diag_best_mrr10:.4f})",
)
for xv, yv in zip(_DIAG_BONUS_GRID, _diag_sweep_mrr):
    ax3.annotate(
        f"{yv:.4f}", xy=(xv, yv), xytext=(0, 7),
        textcoords="offset points", ha="center", fontsize=7.5,
    )
ax3.set_title("Category Bonus Sensitivity (MRR@10)")
ax3.set_xlabel("category_bonus")
ax3.set_ylabel("MRR@10")
ax3.set_ylim(bottom=0)
ax3.legend(fontsize=8)

plt.tight_layout()
_diag_out_path = _DIAG_REPORT_DIR / "cross_encoder_diagnostics.png"
fig.savefig(_diag_out_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\nDiagnostics figure saved to: {_diag_out_path}")
print(f"\n── Diagnostic 1: Score Distribution ──────────────────────────")
print(f"  ROC-AUC        : {_diag_roc_auc:.4f}  (good > 0.85)")
print(f"  Avg Precision  : {_diag_avg_prec:.4f}  (good > 0.80)")
print(f"  Mean pos score : {_diag_mean_pos:.4f}")
print(f"  Mean neg score : {_diag_mean_neg:.4f}")
print(f"  Score gap      : {_diag_score_gap:.4f}  (good > 1.0)")
print(f"\n── Diagnostic 2: MRR Before vs After ─────────────────────────")
print(f"  Embedding MRR@{_DIAG_RETRIEVAL_TOP_K}           : {_diag_mrr_before:.4f}")
print(f"  CE rerank MRR@{_DIAG_RETRIEVAL_TOP_K} (no cat)  : {_diag_mrr_after_ce:.4f}  (delta {_delta_ce:+.4f})")
print(f"  CE rerank MRR@{_DIAG_RETRIEVAL_TOP_K} (+cat)    : {_diag_mrr_after_ce_cat:.4f}  (delta {_delta_cat:+.4f} vs CE alone)")
print(f"\n── Diagnostic 3: Category Bonus Sweep ────────────────────────")
print(f"  Best bonus : {_diag_best_bonus}  →  MRR@10 = {_diag_best_mrr10:.4f}")


## Generate Test Submission

The final submission uses the configured `FINAL_MODEL` and writes the first-stage retrieval ranking directly with `SUBMIT_TOP_K` documents per query.


In [ ]:
final_model_artifacts = prepared_retrievers.get(FINAL_MODEL)
if final_model_artifacts is None:
    final_model_artifacts = prepare_retriever(FINAL_MODEL, docs_df)
    prepared_retrievers[FINAL_MODEL] = final_model_artifacts

test_results = run_retrieval(
    model_name=FINAL_MODEL,
    docs_frame=docs_df,
    queries_frame=test_queries_df,
    top_k=RETRIEVAL_TOP_K,
    prepared_artifacts=final_model_artifacts,
    embedding_kind="queries_test",
)

if ENABLE_CROSS_ENCODER_RERANK:
    if "cross_encoder_reranker" not in globals() or cross_encoder_reranker is None:
        cross_encoder_reranker = build_or_load_cross_encoder(
            train_queries_frame=train_queries_df,
            docs_frame=docs_df,
            ground_truth=ground_truth,
        )
        
    current_bonus = CROSS_ENCODER_CATEGORY_BONUS if ENABLE_CATEGORY_FILTER else 0.0
    
    test_results = rerank_results_with_cross_encoder(
        results=test_results,
        query_frame=test_queries_df,
        docs_frame=docs_df,
        cross_encoder=cross_encoder_reranker,
        query_category_map=test_query_category_map,
        doc_category_map=doc_category_map,
        rerank_top_m=20, 
        category_bonus=current_bonus
    )

write_kaggle_submission(
    test_results,
    sample_csv_path=sample_submission_path,
    output_csv_path=PATHS.output_path,
    category_predictions=test_query_category_map,
)

submission_preview = pd.read_csv(PATHS.output_path)
print(f"Saved submission to: {PATHS.output_path.resolve()}")
print(f"Rows: {len(submission_preview):,}")
submission_preview.head()

## Conclusions

The notebook now reads as a reusable retrieval-engine report rather than a sequence of ad hoc cells.

The key practical takeaway remains the same: **embedding retrieval is the strongest default first-stage method for this dataset** because it is far more robust to semantic mismatch than TF-IDF or BM25+, while the lexical methods still remain useful baselines for exact-match-heavy scenarios and tight latency budgets.
